# Paper #36, original-paper check: the authors' own VisDrone predictions, re-scored
The paper's footnote-1 repo (now github.com/fcakyon/small-object-detection-benchmark, release v0.0.1, pinned) publishes
one `result.json` (COCO detections) and one `eval.json` (their own score) for each VisDrone row of Table 1: FCOS,
VFNet and TOOD with FI, SAHI, SF and overlap (PO). This check downloads all 20 and the official VisDrone2019-DET
test-dev labels (1,610 photos, Ultralytics mirror) and re-scores every file with pycocotools the paper's way (AP at
IoU 0.5, all / small / medium / large, max 500 detections, p.4), next to Table 1 and their eval.json. It also tests
the abstract's gains (SAHI +6.8 / +5.1 / +5.3 AP, with SF +12.7 / +13.4 / +14.5) against the table.
The image ids of their result files follow their own converter's file order, which is not published; the order was
recovered and verified once (c36_visdrone_map.py). CPU only, no training. xView (Table 2) needs a registration.

In [ ]:
# ===== 0. Setup =====================================================================================================
# FAST = True  -> quick test (1 epoch, ~3 % of the images, a few test images): run this first in an interactive
#                 session and check every cell finishes. Then set FAST = False and use Save Version -> Save & Run All.
FAST = False
PAPER = "36"
import os, sys, json, glob, time, subprocess
WORK = os.environ.get("LPR_WORK", "/kaggle/working")
KIT = f"{WORK}/lprkit"; RUNS = f"{WORK}/runs"
os.makedirs(KIT, exist_ok=True); os.makedirs(RUNS, exist_ok=True); sys.path.insert(0, KIT)
os.environ["LPR_WORK"] = WORK
!pip -q install ultralytics==8.4.170 pycocotools
import torch
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
if not os.environ.get("LPR_LOCAL_TEST"):
    assert torch.cuda.device_count() >= 2, "Session options -> Accelerator: GPU T4 x2 (and Internet: On)"
G0, G1 = (0, 1) if torch.cuda.device_count() >= 2 else (0, 0)   # one job per GPU; a single GPU runs them in turn
E = lambda full, fast=1: fast if FAST else full                   # epochs: full run vs quick test
FR = 0.03 if FAST else 1.0                                         # fraction of the training images
TMAX = ["--test-max", "40"] if FAST else []
print("=" * 100)
print("QUICK TEST (FAST = True): 1 epoch on ~3 % of the images. Each job ends after about 1-3 min and the numbers mean"
      " NOTHING;\nit only proves that every cell works. For the real results set FAST = False and Save & Run All."
      if FAST else f"FULL RUN (FAST = False): planned about 0.5 h. A training job that ends after a few"
      " minutes FAILED:\nread the '----- ... failed' lines below the cell and logs/<run>.log.")
print("=" * 100)

## 1. Write the code (`lprkit/`)
The same files are in this folder of the team repo; the cells below only write them to `/kaggle/working/lprkit`.

In [ ]:
%%writefile /kaggle/working/lprkit/common.py
"""Shared helpers for the 498R paper notebooks (Kaggle GPU T4 x2).

  find_dataset()          -> folder that holds dataset_manifest.json (the "498R BanglaLPR All-in-One" dataset)
  data_yaml(part, kind)   -> absolute-path YOLO yaml for detection / recognition (kind 'all' or 'human')
  run_on_gpus(jobs)       -> start one command per GPU at the same time, wait, return exit codes
  env_info(), save_json(), load_json(), timer
"""
import glob
import json
import os
import platform
import subprocess
import sys
import time

WORK = os.environ.get('LPR_WORK', '/kaggle/working')
WORKERS = int(os.environ.get('LPR_WORKERS', 4))     # DataLoader workers (0 for local Windows tests)


# The team copy of the dataset (Google Drive folder "All In One 1.1", shared with the team). v1.0 (runs up to 20 Oct
# 2026) stays in folder "All In One" (1RLwSI5_YINfJr7tymuKBa8Hxkp0dKKSK), file id 1GGVA0Plxxh6uvRuiDVGEOjiWENHXxmv-.
DRIVE_FOLDER = 'https://drive.google.com/drive/folders/11psdzW5Cpr7tXwqzqNxNt3LA-qeDSOQl'
DRIVE_ZIP_ID = '1PPM6ONnkXYkBNJ5DEM94Pa941cBwbDOg'          # BanglaLPR_AllInOne_v1_1.zip (v1.1, 3.63 GB)
DATASET_VERSION = '1.1'                                     # "version" in dataset_manifest.json that the kits expect
DOWNLOAD_DIR = '/tmp/498r_data' if os.path.exists('/kaggle') else os.path.expanduser('~/498r_data')
_version_warned = set()


def _check_version(root):
    """Warn (once per folder) when the dataset found is not the expected version, e.g. an old v1.0 Kaggle input or an
    old unpacked copy, which find_dataset() would otherwise use without a word."""
    try:
        v = json.load(open(os.path.join(root, 'dataset_manifest.json'), encoding='utf-8')).get('version', '?')
    except (OSError, ValueError):
        v = '?'
    if v != DATASET_VERSION and root not in _version_warned:
        _version_warned.add(root)
        print(f'WARNING: the dataset at {root} is version {v}, these notebooks expect v{DATASET_VERSION}. Remove that '
              f'input / folder to download v{DATASET_VERSION} from {DRIVE_FOLDER}, unless you re-run an old result.',
              flush=True)
    return root


def _search():
    for base in ('/kaggle/input', DOWNLOAD_DIR, os.path.expanduser('~/data'), '/content', '.'):
        hits = sorted(glob.glob(os.path.join(base, '**', 'dataset_manifest.json'), recursive=True), key=len)
        if hits:
            return os.path.dirname(hits[0])
    return None


def download_dataset(dest=DOWNLOAD_DIR, file_id=DRIVE_ZIP_ID):
    """Download the dataset zip from the team Google Drive and unpack it (about 3-6 min on Kaggle). It goes to /tmp on
    Kaggle, so the 3.6 GB are NOT saved as notebook output."""
    import shutil
    import urllib.request
    import zipfile
    os.makedirs(dest, exist_ok=True)
    z = os.path.join(dest, 'BanglaLPR_AllInOne.zip')
    url = f'https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t'
    print(f'dataset not attached -> downloading it from the team Drive ({DRIVE_FOLDER})', flush=True)
    for attempt in range(3):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64)'})
            with urllib.request.urlopen(req, timeout=120) as r, open(z, 'wb') as f:
                if 'text/html' in r.headers.get('Content-Type', ''):
                    raise OSError('Google Drive sent a web page instead of the file (download quota reached?)')
                total, done, step = int(r.headers.get('Content-Length') or 0), 0, 0
                while True:
                    b = r.read(1 << 22)
                    if not b:
                        break
                    f.write(b); done += len(b)
                    if done >= step:
                        print(f'  {done / 2 ** 30:.2f} / {total / 2 ** 30:.2f} GB', flush=True); step += 500 * 2 ** 20
            if total and os.path.getsize(z) != total:
                raise OSError('incomplete download')
            break
        except OSError as e:
            print(f'  download try {attempt + 1} failed: {e}', flush=True)
    else:
        raise FileNotFoundError('Could not download the dataset from Google Drive. Attach it instead: Add Input -> '
                                'Datasets -> "498r-bangla-lpr-all-in-one" (or upload the zip from the Drive folder '
                                f'{DRIVE_FOLDER} as a Kaggle dataset).')
    print('  unpacking ...', flush=True)
    with zipfile.ZipFile(z) as f:
        f.extractall(dest)
    os.remove(z)
    root = _search()
    print('dataset ready:', root, flush=True)
    return root


def find_dataset(download=True):
    """The 498R dataset root (folder with dataset_manifest.json): $DATASET_ROOT, an attached Kaggle input, a local copy
    (~/data, /content, ./), else it is downloaded once from the team Google Drive."""
    root = os.environ.get('DATASET_ROOT')
    if root and os.path.exists(os.path.join(root, 'dataset_manifest.json')):
        return _check_version(root)
    root = _search()
    if root:
        return _check_version(root)
    if download and not os.environ.get('LPR_NO_DOWNLOAD'):
        root = download_dataset()
        if root:
            os.environ['DATASET_ROOT'] = root           # jobs started from here inherit it (no second download)
            return _check_version(root)
    raise FileNotFoundError('dataset_manifest.json not found. On Kaggle: Add Input -> "498r-bangla-lpr-all-in-one", '
                            f'or keep Internet on so it downloads from {DRIVE_FOLDER}. Locally: set DATASET_ROOT.')


def data_yaml(part, kind='all', out_dir=None):
    """Copy detection/recognition data(.human).yaml with an absolute `path:` so Ultralytics can read it anywhere."""
    root = find_dataset()
    src = os.path.join(root, part, 'data.yaml' if kind == 'all' else 'data_human.yaml')
    out_dir = out_dir or os.path.join(WORK, 'yaml')
    os.makedirs(out_dir, exist_ok=True)
    lines = open(src, encoding='utf-8').read().splitlines()
    lines = [l for l in lines if not l.startswith('path:') and not l.startswith('#')]
    dst = os.path.join(out_dir, f'{part}_{kind}.yaml')
    with open(dst, 'w', encoding='utf-8') as f:
        f.write(f'path: {os.path.join(root, part)}\n' + '\n'.join(lines) + '\n')
    return dst


def cv2_with_cascade():
    """OpenCV 5 moved the Haar cascades out of the main package; papers #3 and #11 need them. If the installed cv2 has
    no CascadeClassifier, OpenCV 4.x is installed into WORK/opencv4 and imported instead. Call it BEFORE any other
    `import cv2` in the process."""
    probe = subprocess.run([sys.executable, '-c', 'import cv2, sys; sys.exit(0 if hasattr(cv2, "CascadeClassifier") else 1)'])
    if probe.returncode != 0:
        d = os.path.join(WORK, 'opencv4')
        if not os.path.exists(os.path.join(d, 'cv2')):
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--target', d,
                            'opencv-python-headless<5'], check=True)
        sys.path.insert(0, d)
    import cv2
    assert hasattr(cv2, 'CascadeClassifier'), 'this OpenCV has no Haar cascade support'
    return cv2


def gpu_count():
    try:
        import torch
        return torch.cuda.device_count()
    except Exception:
        return 0


def env_info():
    info = {'python': sys.version.split()[0], 'platform': platform.platform(), 'date': time.strftime('%Y-%m-%d %H:%M')}
    try:
        import torch
        info['torch'] = torch.__version__
        info['cuda'] = torch.version.cuda
        info['gpus'] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    except Exception:
        pass
    for mod in ('ultralytics', 'transformers', 'numpy', 'cv2', 'onnxruntime', 'tensorrt', 'timm', 'peft'):
        try:
            m = __import__(mod)
            info[mod] = getattr(m, '__version__', '?')
        except Exception:
            pass
    return info


def save_json(obj, path):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=1, default=str)


def load_json(path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)


def run_on_gpus(jobs, log_dir=None):
    """jobs: list of (name, gpu_id or None, [cmd...], extra_env dict or None). All start together; returns {name: code}.
    Each job gets CUDA_VISIBLE_DEVICES=<gpu_id>, so inside the job the GPU is always device 0.
    Jobs given the same gpu_id run one after the other on that GPU (a queue per GPU)."""
    log_dir = log_dir or os.path.join(WORK, 'logs')
    os.makedirs(log_dir, exist_ok=True)
    codes = {}

    def start(job):
        name, gpu, cmd, extra = (list(job) + [None])[:4]
        env = dict(os.environ)
        if gpu is not None:
            env['CUDA_VISIBLE_DEVICES'] = str(gpu)
        env.update(extra or {})
        log = open(os.path.join(log_dir, f'{name}.log'), 'w', encoding='utf-8')
        print(f'[start] {name} on GPU {gpu}: {" ".join(map(str, cmd))}', flush=True)
        return subprocess.Popen([str(c) for c in cmd], stdout=log, stderr=subprocess.STDOUT, env=env), log, time.time()

    pending = {}
    for job in jobs:
        pending.setdefault(job[1], []).append(job)
    active = {}
    for g in pending:
        if pending[g]:
            job = pending[g].pop(0)
            active[g] = (job,) + start(job)
    while active:
        time.sleep(10)
        for g in list(active):
            job, proc, log, t0 = active[g]
            if proc.poll() is not None:
                log.close()
                codes[job[0]] = proc.returncode
                print(f'[done]  {job[0]} code={proc.returncode} after {(time.time() - t0) / 60:.1f} min', flush=True)
                del active[g]
                if pending[g]:
                    nxt = pending[g].pop(0)
                    active[g] = (nxt,) + start(nxt)
    return codes


def tail(path, n=20):
    try:
        lines = open(path, encoding='utf-8', errors='replace').read().replace('\r', '\n').splitlines()
        return '\n'.join(l for l in lines if l.strip())[-4000:] if n is None else '\n'.join([l for l in lines if l.strip()][-n:])
    except FileNotFoundError:
        return ''


class Timer:
    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *a):
        self.minutes = (time.time() - self.t0) / 60

In [ ]:
%%writefile /kaggle/working/lprkit/vocab.py
"""Canonical Bangla plate vocabulary for the 498R merged recognition set, and the maps from every source's class names.

Every mapping below was checked by eye on crop galleries (logs/peek/cls_kuet.jpg, cls_blpd.jpg, cls_lpdba.jpg) and,
for BLPD, against the English/Bangla ground truth pairs in its external-validation CSV.
All strings are Unicode NFC (NFC keeps য়/ড়/ঢ় decomposed, consistently).
"""
import unicodedata

N = lambda s: unicodedata.normalize('NFC', s)

DIGITS = list('০১২৩৪৫৬৭৮৯')
METRO = 'মেট্রো'
# vehicle-class (series) letters seen on BRTA plates in our sources
LETTERS = ['অ', 'ই', 'উ', 'এ', 'ক', 'খ', 'গ', 'ঘ', 'ঙ', 'চ', 'ছ', 'জ', 'ঝ', 'ট', 'ঠ', 'ড', 'ঢ', 'ত', 'থ', 'দ', 'ধ',
           'ন', 'প', 'ফ', 'ব', 'ভ', 'ম', 'য', 'র', 'ল', 'শ', 'স', 'হ']
# the 64 districts, official Bangla spelling
DISTRICTS = ['ঢাকা', 'গাজীপুর', 'নারায়ণগঞ্জ', 'নরসিংদী', 'মানিকগঞ্জ', 'মুন্সিগঞ্জ', 'টাঙ্গাইল', 'কিশোরগঞ্জ', 'ফরিদপুর',
             'মাদারীপুর', 'গোপালগঞ্জ', 'রাজবাড়ী', 'শরীয়তপুর',
             'চট্টগ্রাম', 'কক্সবাজার', 'রাঙ্গামাটি', 'বান্দরবান', 'খাগড়াছড়ি', 'ফেনী', 'নোয়াখালী', 'লক্ষ্মীপুর', 'কুমিল্লা',
             'চাঁদপুর', 'ব্রাহ্মণবাড়িয়া',
             'রাজশাহী', 'নাটোর', 'নওগাঁ', 'চাঁপাইনবাবগঞ্জ', 'পাবনা', 'সিরাজগঞ্জ', 'বগুড়া', 'জয়পুরহাট',
             'খুলনা', 'বাগেরহাট', 'সাতক্ষীরা', 'যশোর', 'নড়াইল', 'মাগুরা', 'ঝিনাইদহ', 'কুষ্টিয়া', 'চুয়াডাঙ্গা', 'মেহেরপুর',
             'বরিশাল', 'ভোলা', 'পটুয়াখালী', 'বরগুনা', 'ঝালকাঠি', 'পিরোজপুর',
             'সিলেট', 'মৌলভীবাজার', 'হবিগঞ্জ', 'সুনামগঞ্জ',
             'রংপুর', 'দিনাজপুর', 'ঠাকুরগাঁও', 'পঞ্চগড়', 'নীলফামারী', 'লালমনিরহাট', 'কুড়িগ্রাম', 'গাইবান্ধা',
             'ময়মনসিংহ', 'জামালপুর', 'শেরপুর', 'নেত্রকোণা']
# short metro forms printed on plates (Chattogram / Rajshahi metro) and the pre-2014 name of Chapainawabganj
SHORT = ['চট্ট', 'রাজ', 'নবাবগঞ্জ']
CLASSES = [N(x) for x in DIGITS + [METRO] + LETTERS + DISTRICTS + SHORT]
assert len(DISTRICTS) == 64 and len(set(CLASSES)) == len(CLASSES)
CLASS_ID = {c: i for i, c in enumerate(CLASSES)}
KIND = {N(c): ('digit' if c in DIGITS else 'metro' if c == METRO else 'letter' if c in LETTERS else 'district')
        for c in DIGITS + [METRO] + LETTERS + DISTRICTS + SHORT}

# ---- spelling variants -> canonical
VARIANTS = {'নারাইল': 'নড়াইল', 'পটুয়াখালি': 'পটুয়াখালী', 'নীলফামারি': 'নীলফামারী', 'মাদারিপুর': 'মাদারীপুর',
            'শরিয়তপুর': 'শরীয়তপুর', 'চাপাইনবাবগঞ্জ': 'চাঁপাইনবাবগঞ্জ', 'রাঙামাটি': 'রাঙ্গামাটি', 'চট্টো': 'চট্ট',
            'নেত্রকোনা': 'নেত্রকোণা', 'বগুরা': 'বগুড়া', '0': '০'}

# ---- KUET (Hossain & Hassan) English names, checked on cls_kuet.jpg
KUET = {**{str(i): DIGITS[i] for i in range(10)}, 'ka': 'ক', 'kha': 'খ', 'ga': 'গ', 'gha': 'ঘ', 'ca': 'চ', 'ja': 'জ',
        'ta': 'ট', 'da': 'ড', 'na': 'ন', 'ba': 'ব', 'va': 'ভ', 'ha': 'হ', 'metro': METRO, 'dhaka': 'ঢাকা',
        'khulna': 'খুলনা', 'chattogram': 'চট্ট', 'jashore': 'যশোর'}

# ---- BLPD (Sarker & Alam) English names, checked on cls_blpd.jpg and the external-validation CSV
BLPD = {**{str(i): DIGITS[i] for i in range(10)},
        'A': 'অ', 'Ba': 'ব', 'Bha': 'ভ', 'Cha': 'চ', 'Chha': 'ছ', 'DA': 'দ', 'Da': 'ড', 'Dha': 'ঢ', 'E': 'এ',
        'Ga': 'গ', 'Gha': 'ঘ', 'Ha': 'হ', 'Ja': 'জ', 'Jha': 'ঝ', 'Ka': 'ক', 'Kha': 'খ', 'La': 'ল', 'Ma': 'ম',
        'Na': 'ন', 'Pa': 'প', 'Sa': 'স', 'Sha': 'শ', 'THA': 'ঠ', 'Ta': 'ট', 'Tha': 'থ', 'U': 'উ',
        'Metro': METRO, 'Chatto': 'চট্ট', 'Raj': 'রাজ', 'Nowabganj': 'নবাবগঞ্জ',
        'Bagerhat': 'বাগেরহাট', 'Bagura': 'বগুড়া', 'Bandarban': 'বান্দরবান', 'Barguna': 'বরগুনা', 'Barisal': 'বরিশাল',
        'Bhola': 'ভোলা', 'Brahmanbaria': 'ব্রাহ্মণবাড়িয়া', 'Chandpur': 'চাঁদপুর', 'Chapainawabganj': 'চাঁপাইনবাবগঞ্জ',
        'Chattogram': 'চট্টগ্রাম', 'Chuadanga': 'চুয়াডাঙ্গা', 'Cox-s Bazar': 'কক্সবাজার', 'Cox-s-Bazar': 'কক্সবাজার',
        'Cumilla': 'কুমিল্লা', 'Dhaka': 'ঢাকা', 'Dinajpur': 'দিনাজপুর', 'Faridpur': 'ফরিদপুর', 'Feni': 'ফেনী',
        'Gaibandha': 'গাইবান্ধা', 'Gazipur': 'গাজীপুর', 'Gopalganj': 'গোপালগঞ্জ', 'Habiganj': 'হবিগঞ্জ',
        'Jamalpur': 'জামালপুর', 'Jessore': 'যশোর', 'Jhalokati': 'ঝালকাঠি', 'Jhenaidah': 'ঝিনাইদহ',
        'Joypurhat': 'জয়পুরহাট', 'Khagrachari': 'খাগড়াছড়ি', 'Khulna': 'খুলনা', 'Kishoreganj': 'কিশোরগঞ্জ',
        'Kurigram': 'কুড়িগ্রাম', 'Kustia': 'কুষ্টিয়া', 'Lakshmipur': 'লক্ষ্মীপুর', 'Lalmonirhat': 'লালমনিরহাট',
        'Madaripur': 'মাদারীপুর', 'Magura': 'মাগুরা', 'Manikganj': 'মানিকগঞ্জ', 'Meherpur': 'মেহেরপুর',
        'Moulvibazar': 'মৌলভীবাজার', 'Munshiganj': 'মুন্সিগঞ্জ', 'Mymensingh': 'ময়মনসিংহ', 'Naogaon': 'নওগাঁ',
        'Narail': 'নড়াইল', 'Narayanganj': 'নারায়ণগঞ্জ', 'Narsingdi': 'নরসিংদী', 'Natore': 'নাটোর',
        'Netrokona': 'নেত্রকোণা', 'Nilphamari': 'নীলফামারী', 'Noakhali': 'নোয়াখালী', 'Pabna': 'পাবনা',
        'Patuakhali': 'পটুয়াখালী', 'Pirojpur': 'পিরোজপুর', 'Rajbari': 'রাজবাড়ী', 'Rajshahi': 'রাজশাহী',
        'Rangamati': 'রাঙ্গামাটি', 'Rangpur': 'রংপুর', 'Satkhira': 'সাতক্ষীরা', 'Shariatpur': 'শরীয়তপুর',
        'Sherpur': 'শেরপুর', 'Sirajganj': 'সিরাজগঞ্জ', 'Sunamganj': 'সুনামগঞ্জ', 'Sylhet': 'সিলেট',
        'Tangail': 'টাঙ্গাইল', 'Thakurgaon': 'ঠাকুরগাঁও', 'panchagarh': 'পঞ্চগড়'}

# ---- NightLPBD transliterations, checked on cls_night.jpg (syllable boxes are dropped in ingest.py)
NIGHT = {**{str(i): DIGITS[i] for i in range(10)}, 'bo': 'ব', 'co': 'চ', 'go': 'গ', 'ho': 'হ', 'To': 'ট', 'lo': 'ল',
         'Dhaka': 'ঢাকা', 'khulona': 'খুলনা', 'meTrO': METRO, 'zoshOro': 'যশোর'}


# ---- furcifer/bangla-license-plates-synthetic (Kaggle, CC0) lp.names, checked on rendered samples (thaa = থ, daa = দ)
FURCIFER = {**{str(i): DIGITS[i] for i in range(10)}, 'a': 'অ', 'i': 'ই', 'u': 'উ', 'e': 'এ', 'ka': 'ক', 'kha': 'খ',
            'ga': 'গ', 'gha': 'ঘ', 'umno': 'ঙ', 'ca': 'চ', 'cha': 'ছ', 'ja': 'জ', 'jha': 'ঝ', 'tSa': 'ট', 'tSha': 'ঠ',
            'dSa': 'ড', 'dSha': 'ঢ', 'thaa': 'থ', 'daa': 'দ', 'na': 'ন', 'pa': 'প', 'pha': 'ফ', 'ba': 'ব', 'bha': 'ভ',
            'ma': 'ম', 'jSaa': 'য', 'ra': 'র', 'la': 'ল', 'sha': 'শ', 'sa': 'স', 'ha': 'হ', 'dhaka': 'ঢাকা',
            'chittagong': 'চট্টগ্রাম', 'khulna': 'খুলনা', 'kushtia': 'কুষ্টিয়া', 'narayanganj': 'নারায়ণগঞ্জ',
            'bagura': 'বগুড়া', 'comilla': 'কুমিল্লা', 'mymensingh': 'ময়মনসিংহ', 'sylhet': 'সিলেট'}

def canon(source, name):
    """Map a source class name to a canonical class (or None if it cannot be mapped)."""
    if source == 'kuet':
        v = KUET.get(name)
    elif source == 'blpd':
        v = BLPD.get(name)
    elif source == 'nightlpbd':
        v = NIGHT.get(name)
    elif source == 'furcifer':
        v = FURCIFER.get(name)
    else:                                   # lpdba, foggy_bd: Bangla names already
        v = name
    if v is None:
        return None
    v = N(VARIANTS.get(N(v), v))
    v = N(VARIANTS.get(v, v))
    return v if v in CLASS_ID else None


def plate_text(tokens_with_boxes):
    """Plate string from canonical tokens + boxes [[token, x1, y1, x2, y2], ...].
    BRTA layout: top row = district (or short metro form) [মেট্রো] - letter, bottom row = 6 digits (NN-NNNN).
    Rows are taken from the token kind (words vs digits), not from y, so tilted plates still work; y is only used
    to check that the words sit above the digits. Returns (text, valid) where valid = exactly that layout."""
    words = [t for t in tokens_with_boxes if KIND.get(t[0]) == 'district']
    metro = [t for t in tokens_with_boxes if KIND.get(t[0]) == 'metro']
    letters = [t for t in tokens_with_boxes if KIND.get(t[0]) == 'letter']
    digits = [t for t in tokens_with_boxes if KIND.get(t[0]) == 'digit']
    cx = lambda t: (t[1] + t[3]) / 2
    cy = lambda t: (t[2] + t[4]) / 2
    top = ' '.join(t[0] for t in sorted(words, key=cx))
    if metro:
        top = (top + ' ' if top else '') + METRO
    if letters:
        top = (top + '-' if top else '') + ''.join(t[0] for t in sorted(letters, key=cx))
    ds = ''.join(t[0] for t in sorted(digits, key=cx))
    bottom = ds[:2] + '-' + ds[2:] if len(ds) == 6 else ds
    text = (top + ' ' + bottom).strip()
    up = words + metro + letters
    valid = (len(words) == 1 and len(metro) <= 1 and len(letters) == 1 and len(digits) == 6
             and sum(map(cy, up)) / len(up) < sum(map(cy, digits)) / len(digits))
    return text, valid


def norm_text(s):
    """Text used for scoring: NFC, Bangla digits, no spaces/hyphens/dots."""
    s = N(s)
    s = s.translate(str.maketrans('0123456789', '০১২৩৪৫৬৭৮৯'))
    for ch in ' -–—.:,_':
        s = s.replace(ch, '')
    return s


# English transliteration for each canonical class (from the BLPD names, which match BRTA English usage)
ENGLISH = {N(v): k for k, v in BLPD.items() if not k.isdigit() and k not in ('Cox-s-Bazar', 'DA')}
ENGLISH.update({N('দ'): 'DA', N('কক্সবাজার'): "Cox's Bazar", N('ই'): 'I', N('ঙ'): 'Umo', N('ত'): 'Ta (dental)',
                N('ধ'): 'Dha (dental)', N('ফ'): 'Pha', N('য'): 'Ya', N('র'): 'Ra', N('নবাবগঞ্জ'): 'Nawabganj'})
ENGLISH.update({N(d): str(i) for i, d in enumerate(DIGITS)})
ENGLISH[N(METRO)] = 'Metro'


def text_tokens(text):
    """Split a canonical plate text ("ঢাকা মেট্রো-গ ১২-৩৪৫৬") into class tokens: district, [মেট্রো], letter, 6 digits."""
    t = N(text)
    out = []
    head, _, digits = t.rpartition(' ')
    if '-' in head:
        words, _, letter = head.rpartition('-')
    else:
        words, letter = head, ''
    for w in words.split():
        out.append(w)
    if letter:
        out.append(letter)
    out.extend(ch for ch in digits if ch in DIGITS)
    return out


def tokens_to_text(toks):
    """Inverse of text_tokens for a predicted token sequence (any order): district(s) [মেট্রো]-letter DD-DDDD."""
    words = [t for t in toks if KIND.get(t) == 'district']
    metro = [t for t in toks if KIND.get(t) == 'metro']
    letters = [t for t in toks if KIND.get(t) == 'letter']
    digits = ''.join(t for t in toks if KIND.get(t) == 'digit')
    top = ' '.join(words)
    if metro:
        top = (top + ' ' if top else '') + METRO
    if letters:
        top = (top + '-' if top else '') + ''.join(letters)
    bottom = digits[:2] + '-' + digits[2:] if len(digits) == 6 else digits
    return (top + ' ' + bottom).strip()


PUA = 0xE000   # one private-use symbol per token, for libraries that need a 1-character alphabet (PARSeq, fast-plate-ocr)


def sym_alphabet(classes):
    return {c: chr(PUA + i) for i, c in enumerate(classes)}


def text_to_syms(text, classes):
    m = sym_alphabet(classes)
    return ''.join(m[t] for t in text_tokens(text) if t in m)


def syms_to_text(syms, classes):
    inv = {v: k for k, v in sym_alphabet(classes).items()}
    return tokens_to_text([inv[c] for c in syms if c in inv])

In [ ]:
%%writefile /kaggle/working/lprkit/metrics.py
"""OCR scoring and fusion for Bangla plates.

Texts are compared after vocab.norm_text (NFC, Bangla digits, no spaces/hyphens), so "ঢাকা মেট্রো-গ ১২-৩৪৫৬" and
"ঢাকামেট্রোগ১২৩৪৫৬" are the same plate. Three views of one prediction:
  plate exact  : whole plate right
  CER          : Unicode character edit distance / truth length (what papers #1, #32 report)
  token acc    : 1 - token edit distance / truth tokens; tokens = district, মেট্রো, letter, 6 digits (vocab.text_tokens)
  WER          : word edit distance / truth words (words = parts between spaces and hyphens)
"""
import collections

try:
    from . import vocab
except ImportError:
    import vocab


def lev(a, b):
    """Edit distance between two sequences (str or list)."""
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]


def tokens(text):
    t = vocab.N(text or '')
    try:
        return vocab.text_tokens(t)
    except Exception:
        return list(vocab.norm_text(t))


def words(text):
    """Words for WER (papers #1, #32): the plate text split at spaces and hyphens, digits as one word per group."""
    t = vocab.N(text or '')
    for d, b in zip('0123456789', vocab.DIGITS):
        t = t.replace(d, b)
    return t.replace('-', ' ').split()


def score_rows(rows):
    """rows: iterable of (pred_text, true_text). Returns aggregate metrics in %."""
    n = exact = ce = cl = te = tl = we = wl = 0
    for pred, true in rows:
        p, t = vocab.norm_text(pred or ''), vocab.norm_text(true)
        pw, tw = words(pred), words(true)
        we += lev(pw, tw)
        wl += max(1, len(tw))
        n += 1
        exact += int(p == t)
        ce += lev(p, t)
        cl += max(1, len(t))
        pt, tt = tokens(pred), tokens(true)
        te += lev(pt, tt)
        tl += max(1, len(tt))
    return {'plates': n, 'plate_exact_%': round(100 * exact / max(1, n), 2), 'CER_%': round(100 * ce / max(1, cl), 2),
            'char_acc_%': round(max(0.0, 100 * (1 - ce / max(1, cl))), 2), 'token_acc_%': round(max(0.0, 100 * (1 - te / max(1, tl))), 2),
            'WER_%': round(100 * we / max(1, wl), 2)}


def fuse(preds, how='MV-HC', best_model=None):
    """preds: list of (model_name, text, confidence) for ONE plate. Returns the fused text.
    HC: most confident; MV-HC / MV-BM: most frequent normalised string, ties by confidence / by best_model;
    MVCP-HC: vote per token position (tokens padded to the longest), ties by confidence.
    MV-BM+format (ours, not in paper #24): drop readings that break the BRTA layout (is_valid_plate), then MV-BM;
    if no reading fits the layout, MV-BM on all of them."""
    if not preds:
        return ''
    if how == 'HC':
        return max(preds, key=lambda x: x[2])[1]
    if how == 'MV-BM+format':
        return fuse([p for p in preds if is_valid_plate(p[1])] or preds, 'MV-BM', best_model)
    if how.startswith('MV') and not how.startswith('MVCP'):
        groups = collections.defaultdict(list)
        for m, t, c in preds:
            groups[vocab.norm_text(t)].append((m, t, c))
        top = max(len(v) for v in groups.values())
        tied = [v for v in groups.values() if len(v) == top]
        if len(tied) == 1:
            return tied[0][0][1]
        if how == 'MV-BM' and best_model is not None:
            for v in tied:
                if any(m == best_model for m, _, _ in v):
                    return v[0][1]
        return max(tied, key=lambda v: sum(c for _, _, c in v) / len(v))[0][1]
    if how.startswith('MVCP'):
        toks = [(tokens(t), c) for _, t, c in preds]
        L = max(len(t) for t, _ in toks)
        out = []
        for i in range(L):
            votes = collections.defaultdict(list)
            for t, c in toks:
                votes[t[i] if i < len(t) else ''].append(c)
            top = max(len(v) for v in votes.values())
            cands = [(k, max(v)) for k, v in votes.items() if len(v) == top]
            out.append(max(cands, key=lambda kv: kv[1])[0])
        toks_out = [x for x in out if x]
        return ' '.join(toks_out)
    raise ValueError(how)


EXTRA_TESTS = ('test_fog', 'test_rain_synth')     # extra OCR test splits of the 498R dataset (ocr/<name>.csv)


def condition_scores(preds, root):
    """preds: dicts with file / pred / true on the real OCR test split -> scores per condition tag
    (ocr/test_conditions.csv of the 498R dataset: clear, night, blurry, low_res). None if the file is missing."""
    import csv
    import os
    f = os.path.join(root, 'ocr', 'test_conditions.csv')
    if not os.path.exists(f):
        return None
    tags = {r['file']: r for r in csv.DictReader(open(f, encoding='utf-8'))}
    out = {}
    for tag in ('clear', 'night', 'blurry', 'low_res'):
        rows = [(p['pred'], p['true']) for p in preds if tags.get(p['file'], {}).get(tag) == '1']
        if rows:
            out[tag] = score_rows(rows)
    return out


def extra_splits(root):
    import os
    return [s for s in EXTRA_TESTS if os.path.exists(os.path.join(root, 'ocr', f'{s}.csv'))]


def is_valid_plate(text):
    """True if the text parses as one BRTA plate: district [মেট্রো] letter + 6 digits."""
    tk = tokens(text)
    kinds = [vocab.KIND.get(t, '?') for t in tk]
    return (kinds.count('district') == 1 and kinds.count('letter') == 1 and kinds.count('digit') == 6
            and kinds.count('metro') <= 1 and '?' not in kinds)


def mcnemar_exact(b, c):
    """Exact two-sided McNemar test on paired plates: b = plates only system A reads right, c = only system B.
    Binomial test of min(b, c) against Bin(b + c, 0.5). Returns p (1.0 when the two never disagree)."""
    import math
    n = b + c
    if n == 0:
        return 1.0
    p = sum(math.comb(n, i) for i in range(min(b, c) + 1)) / 2 ** n
    return round(min(1.0, 2 * p), 4)

In [ ]:
%%writefile /kaggle/working/lprkit/report.py
"""Write results_vs_paper.md and pack the outputs zip at the end of a paper notebook.

paper_rows: list of (metric label, paper value as text, page, key) where key is a dotted path inside a run's
results.json (e.g. 'test.mAP50', 'ocr_test.rows_decoder.CER_%', 'speed.fp32.fps'); values in 0..1 for
P/R/mAP are shown in %. 'a|b' = the first of a, b with a number (e.g. 'speed.tensorrt_fp16.fps|speed.fp32.fps':
TensorRT, else PyTorch FP32 when the export failed); a fallback value is marked with its source, e.g. '52.90 (fp32)'.
"""
import glob
import json
import os
import shutil
import zipfile

try:
    from . import common
except ImportError:
    import common

PCT_KEYS = ('P', 'R', 'mAP50', 'mAP50-95', 'test_mAP50', 'test_mAP50-95')   # test_*: TensorRT INT8 scores


def get(d, key):
    for k in key.split('.'):
        if not isinstance(d, dict) or k not in d:
            return None
        d = d[k]
    return d


def fmt(v, key):
    if v is None:
        return '–'
    if isinstance(v, (int, float)):
        last = key.split('.')[-1]
        if last in PCT_KEYS and v <= 1.0:
            return f'{100 * v:.1f}'
        if isinstance(v, float):
            return f'{v:.4f}' if 0 < abs(v) < 0.1 else f'{v:.2f}'
        return str(v)
    return str(v)[:60]


def cell(d, key):
    """(text, used a fallback) for key 'a|b|...': the first alternative with a number; if none has one, the first value
    found (e.g. 'failed: ...' when the TensorRT export failed)."""
    alts = key.split('|')
    for i, k in enumerate(alts):
        v = get(d, k)
        if isinstance(v, (int, float)) and not isinstance(v, bool):
            return (fmt(v, k), False) if i == 0 else (f"{fmt(v, k)} ({k.split('.')[-2] if '.' in k else k})", True)
    v = next((x for x in (get(d, k) for k in alts) if x is not None), None)
    return fmt(v, alts[0]), False


def compare_table(runs_dir, paper_rows, runs, title, notes='', same_data=False):
    """key 'run::path' fills only that run's column (for rows that belong to one run)."""
    res = {}
    for r in runs:
        p = os.path.join(runs_dir, r, 'results.json')
        if os.path.exists(p):
            res[r] = common.load_json(p)
    cols = ['Metric', 'Paper (page)'] + list(res)
    lines = [f'# {title}', '', '| ' + ' | '.join(cols) + ' |', '|' + '---|' * len(cols)]
    fallbacks = []
    for label, pval, page, key in paper_rows:
        row = [label, f'{pval} ({page})' if page else str(pval)]
        only, k = key.split('::', 1) if key and '::' in key else (None, key)
        for r, d in res.items():
            if k and (only is None or only == r):
                text, fb = cell(d, k)
                row.append(text)
                if fb and k not in fallbacks:
                    fallbacks.append(k)
            else:
                row.append('')
        lines.append('| ' + ' | '.join(row) + ' |')
    for k in fallbacks:
        first, rest = k.split('|', 1)
        lines += ['', f'A value marked with (...) comes from `{rest}`: that run has no number at `{first}` '
                      '(e.g. the TensorRT export failed or was not run).']
    missing = [r for r in runs if r not in res]
    if missing:
        lines += ['', f'Runs without results.json (failed or not run): {", ".join(missing)}']
    if same_data:
        lines += ['', "Same data and split as the paper (its own published data / weights), so this IS a direct check of",
                  "the paper's numbers. Remaining differences come from software versions, hardware and randomness."]
    else:
        lines += ['', 'The paper used its own data; our numbers are on the 498R All-in-One test split (different data),',
                  'so the comparison shows the direction and size of the gap, not a head-to-head result.']
    if notes:
        lines += ['', notes]
    return '\n'.join(lines) + '\n'


def condition_table(runs_dir, runs):
    """Results by condition (498R condition test sets): detectors -> mAP50 % per subset and the false-alarm rate on
    plate-free photos; OCR models -> plate exact % per tag, on the real fog crops and on synthetic rain copies."""
    det_cols = ['clear', 'night', 'blurry', 'small_plates', 'rain_synth', 'fog_synth']
    det_rows, ocr_rows = [], []
    for r in runs:
        p = os.path.join(runs_dir, r, 'results.json')
        if not os.path.exists(p):
            continue
        d = common.load_json(p)
        c = d.get('test_by_condition')
        if isinstance(c, dict):
            cells = [fmt(get(c, f'{k}.mAP50'), 'mAP50') for k in det_cols]
            det_rows.append(f"| {r} | {fmt(get(d, 'test.mAP50'), 'mAP50')} | " + ' | '.join(cells) +
                            f" | {fmt(get(c, 'no_plate.false_alarm_%'), 'x')} |")
        o = get(d, 'ocr_test')
        if isinstance(o, dict) and (o.get('by_condition') or o.get('extra_tests')):
            b, e = o.get('by_condition') or {}, o.get('extra_tests') or {}
            base = get(o, 'rows_decoder.plate_exact_%')
            ocr_rows.append(f"| {r} | {fmt(base, 'x')} | " + ' | '.join(fmt(get(b, f'{k}.plate_exact_%'), 'x')
                                                                       for k in ('clear', 'night', 'blurry')) +
                            f" | {fmt(get(e, 'test_fog.plate_exact_%'), 'x')} | {fmt(get(e, 'test_rain_synth.plate_exact_%'), 'x')} |")
    out = []
    if det_rows:
        out += ['', '## Results by condition: detectors (test mAP50 %)', '',
                '| Run | all test | clear | night | blurry | small plates | rain (synthetic) | fog (synthetic) | '
                'false alarms on plate-free photos (%) |', '|' + '---|' * 9] + det_rows
    if ocr_rows:
        out += ['', '## Results by condition: plate readers (plate exact %)', '',
                '| Run | all test | clear | night | blurry | fog (real fog copies) | rain (synthetic) |',
                '|' + '---|' * 7] + ocr_rows
    if out:
        out += ['', 'Condition subsets are fixed for the whole team (detection/conditions.json, ocr/conditions.json); '
                    'rain and detection-fog copies are synthetic.']
    return '\n'.join(out) + ('\n' if out else '')


def pack(work, name, extra_globs=()):
    """Zip runs/*/(weights, results.json, args.yaml, results.csv, preds), logs, the md tables and env info."""
    zpath = os.path.join(work, f'{name}_outputs.zip')
    keep = ('results.json', 'args.yaml', 'results.csv', 'ocr_preds_test.csv', 'best.pt', 'last.pt', 'config.json',
            'model.safetensors', 'results.png', 'confusion_matrix.png', 'labels.jpg', 'BoxPR_curve.png')
    with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in glob.glob(os.path.join(work, 'runs', '**', '*'), recursive=True):
            ok_ext = ('.md', '.json', '.pt', '.ckpt', '.onnx', '.safetensors', '.csv', '.yaml')
            if os.path.isfile(f) and (os.path.basename(f) in keep or f.endswith(ok_ext)) and os.path.getsize(f) < 1.5e9:
                if '/eval_' in f.replace('\\', '/') and not f.endswith('.json'):
                    continue
                z.write(f, os.path.relpath(f, work))
        for pat in ('logs/*.log', '*.md', 'environment.json', 'code/*.py') + tuple(extra_globs):
            for f in glob.glob(os.path.join(work, pat)):
                z.write(f, os.path.relpath(f, work))
    print('packed', zpath, round(os.path.getsize(zpath) / 1e6, 1), 'MB')
    # small copy without model files, for the team GitHub repo (weights/ folder; web upload limit 25 MB per file)
    spath = os.path.join(work, f'{name}_results.zip')
    model_ext = ('.pt', '.ckpt', '.onnx', '.safetensors', '.engine', '.bin', '.pth')
    with zipfile.ZipFile(zpath) as src, zipfile.ZipFile(spath, 'w', zipfile.ZIP_DEFLATED) as dst:
        for n in src.namelist():
            if not n.endswith(model_ext) and not n.endswith('/predictions.json'):   # COCO predictions: 10-15 MB each
                dst.writestr(n, src.read(n))
    print('packed', spath, round(os.path.getsize(spath) / 1e6, 1), 'MB (no weights: for GitHub weights/)')
    return zpath

In [ ]:
%%writefile /kaggle/working/lprkit/det.py
"""Train + test one plate DETECTOR (or character detector) with Ultralytics, then write results.json.

  python det.py --data det.yaml --model yolov8s.pt --name v8s --project runs
  python det.py --data det.yaml --model rtdetr-l.pt --rtdetr --optimizer AdamW --lr0 1e-4 --batch 8 --name rtdetr
  python det.py --data det.yaml --name v8s --project runs --fps-only [--trt] [--cpu]
  python det.py ... --stages '[{"epochs":12,"freeze":12,"hyp":{...}}, ...]'      (multi-stage schedule, paper #1)

The test split is used once, at the end, with best.pt (best on val). Speed is measured separately (--fps-only) on an
idle GPU at batch 1 so two trainings running at the same time do not distort it.
results.json: settings, epochs, best epoch, minutes, params, GFLOPs, val/test P R mAP50 mAP50-95, plate counts at
conf 0.25 (found %, correct %, recall by plate size, "IoU>0.7 accuracy" of paper #1), speed, versions.
"""
import argparse
import json
import os
import platform
import sys
import time
from pathlib import Path

import numpy as np
import torch
import yaml

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import common


def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)


def split_files(data_yaml, split):
    d = yaml.safe_load(Path(data_yaml).read_text(encoding='utf-8'))
    src = Path(d['path']) / d[split]
    if src.suffix == '.txt':
        imgs = [Path(d['path']) / l.strip() for l in src.read_text(encoding='utf-8').splitlines() if l.strip()]
        img_dir = imgs[0].parent if imgs else src.parent
    else:
        img_dir = src
        imgs = sorted(p for p in img_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
    lab = lambda p: Path(str(p.parent).replace('images', 'labels', 1)) / (p.stem + '.txt')
    return imgs, lab


def gt_boxes(label_file, w, h, cls=None):
    if not label_file.exists():
        return np.zeros((0, 4)), np.zeros(0, int)
    rows = [l.split() for l in label_file.read_text().splitlines() if l.strip()]
    if not rows:
        return np.zeros((0, 4)), np.zeros(0, int)
    c = np.array([int(r[0]) for r in rows])
    b = np.array([[float(v) for v in r[1:5]] for r in rows])
    xyxy = np.stack([(b[:, 0] - b[:, 2] / 2) * w, (b[:, 1] - b[:, 3] / 2) * h,
                     (b[:, 0] + b[:, 2] / 2) * w, (b[:, 1] + b[:, 3] / 2) * h], 1)
    return xyxy, c


def plate_counts(model, data_yaml, split, imgsz, device, conf=0.25, iou_thr=0.5, max_images=None):
    """Plain-language numbers at one working threshold (conf 0.25, IoU 0.5), plus:
       recall by plate size (COCO areas on the stored image: small < 32^2, medium < 96^2, large) and
       paper #1's metric: share of photos where EVERY plate has a box with IoU > 0.7 (iou70_acc_%), mean IoU."""
    imgs, lab = split_files(data_yaml, split)
    imgs = imgs[:max_images] if max_images else imgs
    tp = fp = fn = ok = 0
    size_tp, size_n = {'small': 0, 'medium': 0, 'large': 0}, {'small': 0, 'medium': 0, 'large': 0}
    iou70 = n_with_gt = 0
    ious = []
    src = {}
    for s in range(0, len(imgs), 32):                 # a list source is one batch in Ultralytics 8.4: feed in chunks
        for r in model.predict([str(p) for p in imgs[s:s + 32]], imgsz=imgsz, conf=conf, device=device, verbose=False):
            h, w = r.orig_shape
            g, _ = gt_boxes(lab(Path(r.path)), w, h)
            p = r.boxes.xyxy.cpu().numpy()
            order = np.argsort(-r.boxes.conf.cpu().numpy())
            m = iou_matrix(p[order], g)
            used, t = {}, 0
            for i in range(len(order)):
                j = int(np.argmax(m[i])) if m.shape[1] else -1
                if j >= 0 and m[i, j] >= iou_thr and j not in used:
                    used[j] = m[i, j]
                    t += 1
            tp, fp, fn = tp + t, fp + len(p) - t, fn + len(g) - t
            sname = Path(r.path).name.split('_')[0]
            sc = src.setdefault(sname, [0, 0, 0])
            sc[0] += t; sc[1] += len(g); sc[2] += 1
            ok += int(t == len(g) and t == len(p))
            for j, gb in enumerate(g):
                a = (gb[2] - gb[0]) * (gb[3] - gb[1])
                k = 'small' if a < 32 ** 2 else 'medium' if a < 96 ** 2 else 'large'
                size_n[k] += 1
                size_tp[k] += int(j in used)
            if len(g):
                n_with_gt += 1
                best = m.max(0) if m.shape[0] else np.zeros(len(g))
                ious.extend(best.tolist())
                iou70 += int(bool(np.all(best > 0.7)))
    return {'images': len(imgs), 'plates': tp + fn, 'found_%': round(100 * tp / max(1, tp + fn), 2),
            'correct_%': round(100 * tp / max(1, tp + fp), 2), 'img_all_ok_%': round(100 * ok / max(1, len(imgs)), 2),
            'recall_by_size_%': {k: (round(100 * size_tp[k] / size_n[k], 2) if size_n[k] else None) for k in size_n},
            'plates_by_size': size_n, 'iou70_acc_%': round(100 * iou70 / max(1, n_with_gt), 2),
            'mean_best_iou_%': round(100 * float(np.mean(ious)), 2) if ious else None, 'conf': conf, 'iou': iou_thr,
            'found_%_by_source': {k: {'found_%': round(100 * v[0] / max(1, v[1]), 2), 'plates': v[1], 'images': v[2]}
                                  for k, v in sorted(src.items())}}


def condition_metrics(model, data_yaml, imgsz, device, run_dir, max_images=None):
    """Test results per condition when the dataset has detection/conditions.json (498R All-in-One):
    real subsets (clear, night, blurry, small_plates) and synthetic rain / fog copies of the whole test split -> mAP;
    no_plate photos -> false-alarm rate (share of plate-free photos with a box at conf 0.25)."""
    y = yaml.safe_load(open(data_yaml, encoding='utf-8'))
    root = Path(y['path'])
    cfile = root / 'conditions.json'
    if not cfile.exists():
        return None
    cond = json.loads(cfile.read_text(encoding='utf-8'))
    out = {}
    tmp = Path(run_dir) / 'cond_yaml'
    tmp.mkdir(parents=True, exist_ok=True)
    jobs = [(k, v['list']) for k, v in cond['real'].items() if k != 'no_plate'] + \
           [(k, 'images/' + v['split']) for k, v in cond['synthetic'].items()]
    for name, test in jobs:
        f = tmp / f'{name}.yaml'
        f.write_text(yaml.safe_dump({**{k: y[k] for k in ('path', 'train', 'val', 'names') if k in y}, 'test': test}),
                     encoding='utf-8')
        try:
            out[name] = val_metrics(model, str(f), 'test', imgsz, device, Path(run_dir) / 'cond', batch=16)
            out[name]['plates'] = plate_counts(model, str(f), 'test', imgsz, device, max_images=max_images)['found_%']
        except Exception as e:                       # e.g. an empty subset
            out[name] = f'skipped: {e}'[:120]
    neg = [l.strip() for l in open(root / cond['real']['no_plate']['list'], encoding='utf-8') if l.strip()]
    neg = [str(root / l[2:]) for l in neg][:max_images]
    if neg:
        hits = 0
        for s in range(0, len(neg), 32):
            hits += sum(len(r.boxes) > 0 for r in model.predict(neg[s:s + 32], imgsz=imgsz, conf=0.25, device=device,
                                                                 verbose=False))
        out['no_plate'] = {'photos': len(neg), 'false_alarm_%': round(100 * hits / len(neg), 2)}
    return out


def val_metrics(model, data_yaml, split, imgsz, device, run_dir, batch=16):
    m = model.val(data=data_yaml, split=split, imgsz=imgsz, batch=batch, device=device, plots=True, workers=common.WORKERS,
                  project=str(run_dir), name=f'eval_{split}', exist_ok=True, verbose=False)
    b = m.box
    out = {'P': round(float(b.mp), 4), 'R': round(float(b.mr), 4), 'mAP50': round(float(b.map50), 4),
           'mAP50-95': round(float(b.map), 4)}
    try:
        out['speed_ms'] = {k: round(v, 2) for k, v in m.speed.items()}
    except Exception:
        pass
    return out


def prec(bits):
    """Ultralytics 8.4 uses quantize=16/32; older versions (e.g. the official YOLOv12 fork) use half=True."""
    try:
        from ultralytics.cfg import DEFAULT_CFG_DICT
        if 'quantize' in DEFAULT_CFG_DICT:
            return {'quantize': bits}
    except Exception:
        pass
    return {'half': bits == 16}


def load_model(weights, rtdetr=False):
    if rtdetr:
        from ultralytics import RTDETR
        return RTDETR(str(weights))
    from ultralytics import YOLO
    return YOLO(str(weights))


def legacy_onnx_export():
    """torch >= 2.9 exports with the new dynamo exporter unless told otherwise (it needs onnxscript); older
    Ultralytics code such as the official YOLOv12 fork does not say, so default it to the TorchScript exporter."""
    import inspect
    import torch.onnx as tox
    if getattr(tox.export, '_lpr_legacy', False) or 'dynamo' not in inspect.signature(tox.export).parameters:
        return
    orig = tox.export

    def export(*args, **kw):
        kw.setdefault('dynamo', False)
        return orig(*args, **kw)
    export._lpr_legacy = True
    tox.export = export


def measure_speed(weights, data_yaml, imgsz, device, rtdetr=False, n=200, trt=False, cpu=False):
    """Batch-1 latency on the test images: preprocess + inference + postprocess (NMS included for YOLOs).
    Per precision: ms_per_image / fps (mean), ms_median / fps_median, slow_calls (timed calls > 3x the median, should
    be 0) and ms_breakdown (median of each part). PyTorch letterboxes every photo to its own shape and the first call
    at a new shape is slow (FP16: ~0.6 s per new shape; it made the old FP16 means up to 8x too slow), so every timed
    photo is run once untimed first, per precision. TensorRT and ONNX use one fixed shape."""
    imgs = [str(p) for p in split_files(data_yaml, 'test')[0][:n]]
    out = {'batch': 1, 'imgsz': imgsz, 'images': len(imgs)}

    def bench(model, dev, kw, k):
        timed = imgs if dev != 'cpu' else imgs[:50]
        for f in timed if dev != 'cpu' else timed[:10]:          # warm-up: every letterbox shape seen once
            model.predict(f, imgsz=imgsz, device=dev, verbose=False, **kw)
        ms, parts = [], []
        for f in timed:
            r = model.predict(f, imgsz=imgsz, device=dev, verbose=False, **kw)[0]
            ms.append(sum(r.speed.values()))
            parts.append(r.speed)
        mean, med = float(np.mean(ms)), float(np.median(ms))
        out[k] = {'ms_per_image': round(mean, 2), 'fps': round(1000 / mean, 1),
                  'ms_median': round(med, 2), 'fps_median': round(1000 / med, 1),
                  'slow_calls': int(sum(m > 3 * med for m in ms)),
                  'ms_breakdown': {kk: round(float(np.median([p[kk] for p in parts])), 2) for kk in parts[0]}}

    if torch.cuda.is_available() and not cpu:
        out['device'] = torch.cuda.get_device_name(0)
        for bits in (32, 16):                           # Ultralytics 8.4: quantize=16/32 replaces half=True
            bench(load_model(weights, rtdetr), device, prec(bits), f'fp{bits}')
        if trt:
            try:
                eng = load_model(weights, rtdetr).export(format='engine', imgsz=imgsz, device=device, **prec(16))
                bench(load_model(eng, rtdetr), device, {}, 'tensorrt_fp16')
            except Exception as e:
                out['tensorrt_fp16'] = f'failed: {e}'[:300]
        if trt == 'int8':
            try:
                kw = {'quantize': 8} if 'quantize' in prec(16) else {'int8': True}
                eng = load_model(weights, rtdetr).export(format='engine', imgsz=imgsz, device=device, data=data_yaml,
                                                         fraction=0.3, **kw)
                m8 = load_model(eng, rtdetr)
                bench(m8, device, {}, 'tensorrt_int8')
                out['tensorrt_int8']['test_mAP50'] = round(float(m8.val(data=data_yaml, split='test', imgsz=imgsz, workers=common.WORKERS,
                                                           batch=1, device=device, verbose=False).box.map50), 4)
            except Exception as e:
                out['tensorrt_int8'] = f'failed: {e}'[:300]
    try:
        out['cpu_device'] = platform.processor() or 'cpu'
        legacy_onnx_export()
        onnx = load_model(weights, rtdetr).export(format='onnx', imgsz=imgsz, simplify=False, dynamic=False)
        bench(load_model(onnx, rtdetr), 'cpu', {}, 'cpu_onnx')
    except Exception as e:
        out['cpu_onnx'] = f'failed: {e}'[:300]
    return out


def add_time_cap(model, hours):
    """Safety net for the 12 h Kaggle limit: stop after the epoch that passes `hours` (epochs stay as set, unlike
    Ultralytics' own `time`, which re-plans the epoch count)."""
    if not hours:
        return
    t0 = time.time()

    def cb(trainer):
        if time.time() - t0 > hours * 3600:
            trainer.stop = True
            print(f'[time cap] {hours} h reached after epoch {trainer.epoch + 1}: stopping', flush=True)
    model.add_callback('on_fit_epoch_end', cb)


def train_once(a, weights, name, epochs, extra):
    model = load_model(weights, a.rtdetr)
    add_time_cap(model, a.max_hours)
    kw = dict(data=a.data, epochs=epochs, imgsz=a.imgsz, batch=a.batch, device=a.device, workers=a.workers,
              seed=a.seed, patience=a.patience, optimizer=a.optimizer, cache=a.cache if a.cache != 'none' else False,
              fraction=a.fraction, project=a.project, name=name, exist_ok=True, plots=True, verbose=True)
    if a.lr0 is not None:
        kw['lr0'] = a.lr0
    if a.time:
        kw['time'] = a.time
    kw.update(extra)
    model.train(**kw)
    return Path(a.project) / name


def main():
    p = argparse.ArgumentParser()
    p.add_argument('--data', required=True)
    p.add_argument('--model', default='yolov8s.pt')
    p.add_argument('--rtdetr', action='store_true')
    p.add_argument('--name', required=True)
    p.add_argument('--project', required=True)
    p.add_argument('--epochs', type=int, default=100)
    p.add_argument('--imgsz', type=int, default=640)
    p.add_argument('--batch', type=int, default=16)
    p.add_argument('--device', default='0')
    p.add_argument('--workers', type=int, default=common.WORKERS)
    p.add_argument('--seed', type=int, default=42)
    p.add_argument('--patience', type=int, default=100)
    p.add_argument('--optimizer', default='SGD', help="SGD by default: Ultralytics 8.4 'auto' picks MuSGD (YOLO26)")
    p.add_argument('--lr0', type=float, default=None)
    p.add_argument('--time', type=float, default=None, help='hours; caps training time (Ultralytics adjusts epochs)')
    p.add_argument('--max-hours', type=float, default=None, help='stop after this many hours (keeps the epoch plan)')
    p.add_argument('--cache', default='none')
    p.add_argument('--fraction', type=float, default=1.0, help='<1 only for the quick test')
    p.add_argument('--hyp', default='{}', help='JSON of extra Ultralytics train arguments (augmentation etc.)')
    p.add_argument('--stages', default='', help='JSON list of {epochs, freeze, hyp}: trained one after another')
    p.add_argument('--labels', default='human+auto', help='which labels were used (written to results.json)')
    p.add_argument('--note', default='')
    p.add_argument('--fps-only', action='store_true')
    p.add_argument('--trt', nargs='?', const='fp16', default='', help='also TensorRT FP16 (--trt) or FP16+INT8 (--trt int8)')
    p.add_argument('--cpu', action='store_true')
    p.add_argument('--test-max', type=int, default=None, help='quick test: score only N test images')
    a = p.parse_args()

    from ultralytics import __version__ as ul_version
    run = Path(a.project) / a.name
    res_file = run / 'results.json'
    if a.fps_only:
        r = json.loads(res_file.read_text())
        r['speed'] = measure_speed(run / 'weights' / 'best.pt', a.data, a.imgsz, a.device, a.rtdetr, trt=a.trt, cpu=a.cpu)
        res_file.write_text(json.dumps(r, indent=2))
        print(a.name, 'speed:', json.dumps(r['speed']))
        return

    hyp = json.loads(a.hyp)
    t0 = time.time()
    stage_runs = []
    if a.stages:
        weights = a.model
        for i, st in enumerate(json.loads(a.stages)):
            extra = dict(hyp, **st.get('hyp', {}))
            if 'freeze' in st:
                extra['freeze'] = st['freeze']
            sname = f'{a.name}_stage{i + 1}'
            rd = train_once(a, weights, sname, st['epochs'], extra)
            weights = rd / 'weights' / 'best.pt'
            stage_runs.append(sname)
        run.mkdir(parents=True, exist_ok=True)
        (run / 'weights').mkdir(exist_ok=True)
        import shutil
        shutil.copy2(weights, run / 'weights' / 'best.pt')
        shutil.copy2(Path(a.project) / stage_runs[-1] / 'weights' / 'last.pt', run / 'weights' / 'last.pt')
        shutil.copy2(Path(a.project) / stage_runs[-1] / 'args.yaml', run / 'args.yaml')
        import pandas as pd
        logs = []
        for i, sname in enumerate(stage_runs):
            df = pd.read_csv(Path(a.project) / sname / 'results.csv')
            df.columns = [c.strip() for c in df.columns]
            df['stage'] = i + 1
            logs.append(df)
        log = pd.concat(logs, ignore_index=True)
        log['epoch'] = range(1, len(log) + 1)
        log.to_csv(run / 'results.csv', index=False)
    else:
        train_once(a, a.model, a.name, a.epochs, hyp)
        import pandas as pd
        log = pd.read_csv(run / 'results.csv')
        log.columns = [c.strip() for c in log.columns]
    train_min = (time.time() - t0) / 60
    key50 = [c for c in log.columns if 'mAP50(B)' in c][0]
    key95 = [c for c in log.columns if 'mAP50-95(B)' in c][0]
    fit = 0.1 * log[key50] + 0.9 * log[key95]

    best = load_model(run / 'weights' / 'best.pt', a.rtdetr)
    try:
        from ultralytics.utils.torch_utils import get_flops, get_num_params
        n_params, gflops = get_num_params(best.model), get_flops(best.model, imgsz=a.imgsz)
    except Exception:
        n_params, gflops = 0, 0
    r = {
        'run': a.name, 'labels': a.labels, 'note': a.note, 'stages': stage_runs,
        'settings': {**yaml.safe_load((run / 'args.yaml').read_text()), 'start_weights': a.model, 'hyp': hyp,
                     'stages': json.loads(a.stages) if a.stages else None},
        'epochs_planned': a.epochs, 'epochs_run': int(len(log)), 'best_epoch': int(log['epoch'].iloc[int(fit.idxmax())]),
        'train_minutes': round(train_min, 1), 'params_M': round(n_params / 1e6, 2), 'GFLOPs': round(float(gflops), 1),
        'val': val_metrics(best, a.data, 'val', a.imgsz, a.device, run),
        'test': val_metrics(best, a.data, 'test', a.imgsz, a.device, run),
        'test_plates': plate_counts(best, a.data, 'test', a.imgsz, a.device, max_images=a.test_max),
        'test_by_condition': condition_metrics(best, a.data, a.imgsz, a.device, run, max_images=a.test_max),
        'versions': {'ultralytics': ul_version, 'torch': torch.__version__, 'python': platform.python_version()},
        'train_device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'peak_gpu_mem_GB': round(torch.cuda.max_memory_reserved() / 2 ** 30, 2) if torch.cuda.is_available() else None,
    }
    res_file.write_text(json.dumps(r, indent=2))
    print('DONE', a.name, '| test:', r['test'], '| plates:', r['test_plates'], f"| {r['train_minutes']} min")


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /kaggle/working/lprkit/rec.py
"""Character-level YOLO OCR ("detect every Bangla token as an object", paper #2 style): train, test, read plates.

  python rec.py --data rec.yaml --model yolov8n.pt --name chars_v8n --project runs --epochs 100 --imgsz 320
  python rec.py --data rec.yaml --name chars_v8n --project runs --ocr-only        (score the OCR again)

After training: character-box mAP on the recognition test split, then every plate crop of ocr/test.csv is read:
tokens (district, মেট্রো, letter, digits) are joined two ways and scored against the human text:
  rows : words above, digits below (vocab.plate_text)  - our decoder
  xsort: all tokens left to right (what paper #2 describes)
Writes results.json (+ ocr_preds_test.csv with one line per plate for fusion, paper #24).
"""
import argparse
import csv
import json
import os
import platform
import sys
import time
from pathlib import Path

import numpy as np
import torch

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import common
import metrics
import vocab
from det import add_time_cap, val_metrics


def read_csv(path):
    with open(path, encoding='utf-8') as f:
        return list(csv.DictReader(f))


def decode(names, xyxy, conf, cls, min_conf=0.25):
    keep = [(names[int(c)], b, float(s)) for b, s, c in zip(xyxy, conf, cls) if s >= min_conf]
    toks = [[n] + list(map(float, b)) for n, b, s in keep]
    text, ok = vocab.plate_text(toks)
    xs = ' '.join(n for n, b, s in sorted(keep, key=lambda k: (k[1][0] + k[1][2]) / 2))
    mean_conf = float(np.mean([s for _, _, s in keep])) if keep else 0.0
    return text, xs, ok, mean_conf


def read_plates(weights, split='test', imgsz=320, device='0', max_n=None, rtdetr=False):
    from ultralytics import YOLO
    root = common.find_dataset()
    rows = read_csv(os.path.join(root, 'ocr', f'{split}.csv'))[:max_n]
    model = YOLO(str(weights))
    names = model.names
    out, ms = [], []
    paths = [os.path.join(root, 'ocr', 'images', split, r['file']) for r in rows]
    for s in range(0, len(paths), 64):
        for r, res in zip(rows[s:s + 64], model.predict(paths[s:s + 64], imgsz=imgsz, conf=0.10, device=device,
                                                         verbose=False)):
            b = res.boxes
            text, xs, ok, mc = decode(names, b.xyxy.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy())
            out.append({'file': r['file'], 'true': r['text'], 'pred': text, 'pred_xsort': xs, 'valid': ok,
                        'conf': round(mc, 4), 'source': r['source']})
    for p in paths[:100]:                                       # batch-1 latency per plate
        res = model.predict(p, imgsz=imgsz, device=device, verbose=False)[0]
        ms.append(sum(res.speed.values()))
    return out, float(np.mean(ms[10:])) if len(ms) > 10 else float(np.mean(ms or [0]))


def ocr_report(preds, ms):
    rep = {'rows_decoder': metrics.score_rows((p['pred'], p['true']) for p in preds),
           'xsort_decoder': metrics.score_rows((p['pred_xsort'], p['true']) for p in preds),
           'valid_format_%': round(100 * np.mean([p['valid'] for p in preds]), 2) if preds else 0,
           'ms_per_plate': round(ms, 2)}
    by_src = {}
    for src in sorted({p['source'] for p in preds}):
        by_src[src] = metrics.score_rows((p['pred'], p['true']) for p in preds if p['source'] == src)
    rep['by_source'] = by_src
    return rep


def main():
    p = argparse.ArgumentParser()
    p.add_argument('--data', required=True)
    p.add_argument('--model', default='yolov8n.pt')
    p.add_argument('--name', required=True)
    p.add_argument('--project', required=True)
    p.add_argument('--epochs', type=int, default=100)
    p.add_argument('--imgsz', type=int, default=320)
    p.add_argument('--batch', type=int, default=32)
    p.add_argument('--device', default='0')
    p.add_argument('--workers', type=int, default=common.WORKERS)
    p.add_argument('--seed', type=int, default=42)
    p.add_argument('--optimizer', default='SGD')
    p.add_argument('--lr0', type=float, default=None)
    p.add_argument('--time', type=float, default=None)
    p.add_argument('--max-hours', type=float, default=None)
    p.add_argument('--fraction', type=float, default=1.0)
    p.add_argument('--hyp', default='{}')
    p.add_argument('--ocr-only', action='store_true')
    p.add_argument('--ocr-max', type=int, default=None)
    p.add_argument('--note', default='')
    a = p.parse_args()

    from ultralytics import YOLO, __version__ as ul_version
    run = Path(a.project) / a.name
    res_file = run / 'results.json'
    if not a.ocr_only:
        hyp = {'fliplr': 0.0, 'mosaic': 0.5, 'degrees': 3.0, 'shear': 2.0, 'scale': 0.3}   # never mirror text
        hyp.update(json.loads(a.hyp))
        model = YOLO(a.model)
        add_time_cap(model, a.max_hours)
        t0 = time.time()
        kw = dict(data=a.data, epochs=a.epochs, imgsz=a.imgsz, batch=a.batch, device=a.device, workers=a.workers,
                  seed=a.seed, optimizer=a.optimizer, fraction=a.fraction, project=a.project, name=a.name,
                  exist_ok=True, plots=True, verbose=True, patience=100, **hyp)
        if a.lr0 is not None:
            kw['lr0'] = a.lr0
        if a.time:
            kw['time'] = a.time
        model.train(**kw)
        import pandas as pd
        log = pd.read_csv(run / 'results.csv')
        log.columns = [c.strip() for c in log.columns]
        best = YOLO(run / 'weights' / 'best.pt')
        r = {'run': a.name, 'kind': 'char-yolo', 'note': a.note,
             'settings': {'start_weights': a.model, 'epochs': a.epochs, 'imgsz': a.imgsz, 'batch': a.batch,
                          'optimizer': a.optimizer, 'hyp': hyp},
             'epochs_planned': a.epochs, 'epochs_run': int(len(log)), 'train_minutes': round((time.time() - t0) / 60, 1),
             'val_boxes': val_metrics(best, a.data, 'val', a.imgsz, a.device, run, batch=32),
             'test_boxes': val_metrics(best, a.data, 'test', a.imgsz, a.device, run, batch=32),
             'versions': {'ultralytics': ul_version, 'torch': torch.__version__, 'python': platform.python_version()},
             'train_device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}
    else:
        r = json.loads(res_file.read_text(encoding='utf-8'))
    preds, ms = read_plates(run / 'weights' / 'best.pt', 'test', a.imgsz, a.device, a.ocr_max)
    r['ocr_test'] = ocr_report(preds, ms)
    root = common.find_dataset()
    r['ocr_test']['by_condition'] = metrics.condition_scores(preds, root)
    r['ocr_test']['extra_tests'] = {}
    for sp in metrics.extra_splits(root):            # real fog crops, synthetic rain copies
        ep, _ = read_plates(run / 'weights' / 'best.pt', sp, a.imgsz, a.device, a.ocr_max)
        r['ocr_test']['extra_tests'][sp] = metrics.score_rows((p['pred'], p['true']) for p in ep)
    vpreds, _ = read_plates(run / 'weights' / 'best.pt', 'val', a.imgsz, a.device, a.ocr_max)
    r['ocr_val'] = metrics.score_rows((p['pred'], p['true']) for p in vpreds)
    for name, rows in (('test', preds), ('val', vpreds)):
        with open(run / f'ocr_preds_{name}.csv', 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader()
            w.writerows(rows)
    res_file.write_text(json.dumps(r, indent=2, ensure_ascii=False), encoding='utf-8')
    print('DONE', a.name, '| boxes:', r.get('test_boxes'), '| OCR:', r['ocr_test']['rows_decoder'])


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /kaggle/working/lprkit/seq.py
"""Sequence OCR on plate crops (ocr/{split}.csv): CRNN-CTC or a small ViTSTR, trained from the human plate texts.

  python seq.py --model crnn   --name crnn   --project runs --epochs 60
  python seq.py --model vitstr --name vitstr --project runs --epochs 60
  python seq.py ... --train-extra synth.csv      (extra training rows: file paths + text, e.g. synthetic plates)

Labels are TOKENS (district, মেট্রো, letter, digits; vocab.text_tokens), one class each, so Bangla conjuncts and
vowel signs never have to be spelled. Two-line plates: the crop is cut into top and bottom halves that are placed side
by side ("stack" = rows -> one line, so a left-to-right CTC can read the top row then the bottom row).
Writes results.json + ocr_preds_test.csv (for fusion, paper #24).
"""
import argparse
import csv
import json
import math
import os
import random
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import common
import metrics
import vocab


def load_rows(split, extra=None):
    root = common.find_dataset()
    rows = []
    with open(os.path.join(root, 'ocr', f'{split}.csv'), encoding='utf-8') as f:
        for r in csv.DictReader(f):
            rows.append({'path': os.path.join(root, 'ocr', 'images', split, r['file']), 'text': r['text'],
                         'source': r['source'], 'file': r['file']})
    if extra:
        with open(extra, encoding='utf-8') as f:
            for r in csv.DictReader(f):
                rows.append({'path': r['path'], 'text': r['text'], 'source': r.get('source', 'extra'), 'file': r['path']})
    return rows


def class_list():
    root = common.find_dataset()
    return json.load(open(os.path.join(root, 'vocab.json'), encoding='utf-8'))['classes']


def prep(img, H, W, stack, train, flip=False):
    """grey, optional augment, two-line stacking, aspect-kept resize + pad to H x W, float tensor."""
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    if train:
        h, w = g.shape
        ang = random.uniform(-4, 4)
        M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, random.uniform(0.92, 1.05))
        M[:, 2] += [random.uniform(-0.04, 0.04) * w, random.uniform(-0.05, 0.05) * h]
        g = cv2.warpAffine(g, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
        g = np.clip(g.astype(np.float32) * random.uniform(0.6, 1.3) + random.uniform(-30, 30), 0, 255).astype(np.uint8)
        if random.random() < 0.3:
            k = random.choice([3, 5]); g = cv2.GaussianBlur(g, (k, k), 0)
        if random.random() < 0.3:
            g = np.clip(g + np.random.normal(0, random.uniform(3, 12), g.shape), 0, 255).astype(np.uint8)
        if random.random() < 0.2:
            s = random.uniform(0.35, 0.7)
            g = cv2.resize(cv2.resize(g, None, fx=s, fy=s), (g.shape[1], g.shape[0]))
        if flip:
            g = g[:, ::-1]
    h, w = g.shape
    if stack and w / max(1, h) < 2.6:                 # two-line plate -> one line
        top, bot = g[: h // 2], g[h // 2:]
        g = np.concatenate([top, cv2.resize(bot, (w, top.shape[0]))], 1)
        h, w = g.shape
    s = min(H / h, W / w)
    nh, nw = max(1, int(h * s)), max(1, int(w * s))
    canvas = np.full((H, W), 127, np.uint8)
    y0, x0 = (H - nh) // 2, (W - nw) // 2
    canvas[y0:y0 + nh, x0:x0 + nw] = cv2.resize(g, (nw, nh), interpolation=cv2.INTER_AREA)
    return torch.from_numpy(canvas).float().div(255).sub(0.5).div(0.5)[None]


class DS(torch.utils.data.Dataset):
    def __init__(self, rows, cls, H, W, stack, train, flip=False):
        self.rows, self.idx, self.H, self.W, self.stack, self.train, self.flip = rows, {c: i + 1 for i, c in enumerate(cls)}, H, W, stack, train, flip

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        img = cv2.imread(r['path'])
        if img is None:
            img = np.zeros((40, 120, 3), np.uint8)
        x = prep(img, self.H, self.W, self.stack, self.train, self.flip and random.random() < 0.5)
        y = [self.idx[t] for t in vocab.text_tokens(r['text']) if t in self.idx]
        return x, torch.tensor(y, dtype=torch.long), i


def collate(b):
    xs, ys, ids = zip(*b)
    return torch.stack(xs), list(ys), list(ids)


class CRNN(nn.Module):
    """VGG-style CNN -> height collapsed -> 2-layer BiLSTM -> per-column classes (CTC, class 0 = blank)."""
    def __init__(self, n_cls, H=48):
        super().__init__()
        def blk(i, o, pool):
            return [nn.Conv2d(i, o, 3, 1, 1), nn.BatchNorm2d(o), nn.ReLU(True)] + ([nn.MaxPool2d(pool)] if pool else [])
        self.cnn = nn.Sequential(*blk(1, 64, (2, 2)), *blk(64, 128, (2, 2)), *blk(128, 256, None),
                                 *blk(256, 256, (2, 1)), *blk(256, 384, None), *blk(384, 384, (2, 1)))
        self.rnn = nn.LSTM(384 * (H // 16), 256, num_layers=2, bidirectional=True, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(512, n_cls + 1)

    def forward(self, x):
        f = self.cnn(x)                               # B, C, H/16, W/4
        b, c, h, w = f.shape
        f = f.permute(0, 3, 1, 2).reshape(b, w, c * h)
        return self.fc(self.rnn(f)[0])                # B, T, n+1


class ViTSTR(nn.Module):
    """ViTSTR-style: a small pretrained ViT (timm) reads the image; the first L output tokens are classified
    (position i -> token i of the plate, 0 = end/padding)."""
    def __init__(self, n_cls, H, W, L=12, name='vit_tiny_patch16_224'):
        super().__init__()
        import timm
        self.vit = timm.create_model(name, pretrained=True, img_size=(H, W), in_chans=1, num_classes=0)
        self.L = L
        self.head = nn.Linear(self.vit.num_features, n_cls + 1)

    def forward(self, x):
        f = self.vit.forward_features(x)              # B, 1+N, D
        return self.head(f[:, 1:self.L + 1])          # B, L, n+1


class SlotCNN(nn.Module):
    """fast-plate-ocr style (papers #18, #24): plain CNN on a 64x128 grey crop, then L fixed character slots, each a
    softmax over the alphabet (0 = empty slot). No CTC, no attention: very fast."""
    def __init__(self, n_cls, L=9, width=1.0):
        super().__init__()
        c = lambda x: int(x * width)
        def blk(i, o):
            return [nn.Conv2d(i, o, 3, 1, 1), nn.BatchNorm2d(o), nn.ReLU(True)]
        self.f = nn.Sequential(*blk(1, c(32)), nn.MaxPool2d(2), *blk(c(32), c(64)), nn.MaxPool2d(2),
                               *blk(c(64), c(128)), *blk(c(128), c(128)), nn.MaxPool2d(2),
                               *blk(c(128), c(256)), *blk(c(256), c(256)), nn.MaxPool2d(2),
                               nn.Conv2d(c(256), c(512), 1), nn.ReLU(True))
        self.L, self.n = L, n_cls + 1
        self.head = nn.Conv2d(c(512), L * (n_cls + 1), 1)

    def forward(self, x):
        y = self.head(self.f(x))                         # B, L*(n+1), h, w
        y = y.amax(dim=(2, 3))                           # global max pool (fast-plate-ocr)
        return y.view(x.shape[0], self.L, self.n)


def ctc_decode(logits):
    best = logits.argmax(-1).cpu().numpy()
    out, conf = [], []
    probs = logits.softmax(-1).max(-1).values.cpu().numpy()
    for seq, pr in zip(best, probs):
        toks, cs, prev = [], [], 0
        for k, (c, p) in enumerate(zip(seq, pr)):
            if c != 0 and c != prev:
                toks.append(int(c)); cs.append(float(p))
            prev = c
        out.append(toks); conf.append(float(np.mean(cs)) if cs else 0.0)
    return out, conf


def fixed_decode(logits):
    best = logits.argmax(-1).cpu().numpy()
    probs = logits.softmax(-1).max(-1).values.cpu().numpy()
    out, conf = [], []
    for seq, pr in zip(best, probs):
        toks, cs = [], []
        for c, p in zip(seq, pr):
            if c == 0:
                break
            toks.append(int(c)); cs.append(float(p))
        out.append(toks); conf.append(float(np.mean(cs)) if cs else 0.0)
    return out, conf


def run_epoch(model, dl, opt, sched, kind, device, scaler, train):
    model.train(train)
    tot, n = 0.0, 0
    ctc = nn.CTCLoss(blank=0, zero_infinity=True)
    for x, ys, _ in dl:
        x = x.to(device)
        with torch.autocast('cuda', enabled=device.type == 'cuda'):
            out = model(x)
        out = out.float()
        if kind == 'crnn':
            lp = out.log_softmax(-1).permute(1, 0, 2)
            tgt = torch.cat([y for y in ys]).to(device)
            loss = ctc(lp, tgt, torch.full((x.shape[0],), lp.shape[0], dtype=torch.long),
                       torch.tensor([len(y) for y in ys], dtype=torch.long))
        else:
            L = out.shape[1]
            tgt = torch.zeros((x.shape[0], L), dtype=torch.long)
            for i, y in enumerate(ys):
                tgt[i, :min(L, len(y))] = y[:L]
            loss = F.cross_entropy(out.reshape(-1, out.shape[-1]), tgt.to(device).reshape(-1), label_smoothing=0.05)
        if train:
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt); scaler.update(); sched.step()
        tot += float(loss.detach()) * x.shape[0]; n += x.shape[0]
    return tot / max(1, n)


@torch.no_grad()
def predict(model, rows, cls, kind, a, device):
    model.eval()
    ds = DS(rows, cls, a.H, a.W, a.stack, False)
    dl = torch.utils.data.DataLoader(ds, batch_size=64, shuffle=False, num_workers=min(2, common.WORKERS), collate_fn=collate)
    preds = []
    for x, ys, ids in dl:
        out = model(x.to(device)).float()
        toks, conf = ctc_decode(out) if kind == 'crnn' else fixed_decode(out)
        for i, t, c in zip(ids, toks, conf):
            text = vocab.tokens_to_text([cls[k - 1] for k in t])
            r = rows[i]
            preds.append({'file': r['file'], 'true': r['text'], 'pred': text, 'conf': round(c, 4), 'source': r['source']})
    # batch-1 latency
    ms = []
    for r in rows[:120]:
        x = ds[rows.index(r)][0][None].to(device)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(x)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        ms.append(1000 * (time.perf_counter() - t0))
    return preds, float(np.mean(ms[20:])) if len(ms) > 20 else 0.0


def main():
    p = argparse.ArgumentParser()
    p.add_argument('--model', default='crnn', choices=['crnn', 'vitstr', 'slot'])
    p.add_argument('--name', required=True)
    p.add_argument('--project', required=True)
    p.add_argument('--epochs', type=int, default=60)
    p.add_argument('--batch', type=int, default=64)
    p.add_argument('--lr', type=float, default=None)
    p.add_argument('--H', type=int, default=48)
    p.add_argument('--W', type=int, default=256)
    p.add_argument('--no-stack', dest='stack', action='store_false')
    p.add_argument('--flip', action='store_true', help='random horizontal flip (only to show why it is wrong)')
    p.add_argument('--train-extra', default='', help='csv with path,text[,source] rows added to training')
    p.add_argument('--real-fraction', type=float, default=1.0, help='use only this fraction of the real train rows')
    p.add_argument('--seed', type=int, default=42)
    p.add_argument('--max-test', type=int, default=None)
    p.add_argument('--note', default='')
    a = p.parse_args()
    random.seed(a.seed); np.random.seed(a.seed); torch.manual_seed(a.seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    run = Path(a.project) / a.name
    run.mkdir(parents=True, exist_ok=True)
    cls = class_list()
    tr = load_rows('train')
    if a.real_fraction < 1:
        random.Random(a.seed).shuffle(tr)
        tr = tr[:max(1, int(len(tr) * a.real_fraction))]
    n_real = len(tr)
    if a.train_extra:
        tr += load_rows('train', a.train_extra)[len(load_rows('train')):]
    va, te = load_rows('val'), load_rows('test')[:a.max_test]
    if a.model == 'crnn':
        model, lr = CRNN(len(cls), a.H), a.lr or 1e-3
    elif a.model == 'slot':
        a.H, a.W = (64, 128) if a.H == 48 else (a.H, a.W)
        a.stack = False                                   # the CNN sees the whole two-line plate
        model, lr = SlotCNN(len(cls)), a.lr or 1e-3
    else:
        a.H, a.W = (32, 256) if a.H == 48 else (a.H, a.W)
        model, lr = ViTSTR(len(cls), a.H, a.W), a.lr or 3e-4
    model.to(device)
    dl = torch.utils.data.DataLoader(DS(tr, cls, a.H, a.W, a.stack, True, a.flip), batch_size=a.batch, shuffle=True,
                                     num_workers=common.WORKERS, collate_fn=collate, drop_last=True)
    dv = torch.utils.data.DataLoader(DS(va, cls, a.H, a.W, a.stack, False), batch_size=a.batch, num_workers=min(2, common.WORKERS),
                                     collate_fn=collate)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=a.epochs * max(1, len(dl)), pct_start=0.1)
    scaler = torch.amp.GradScaler(enabled=device.type == 'cuda')
    t0, best, log = time.time(), 1e9, []
    for ep in range(a.epochs):
        tl = run_epoch(model, dl, opt, sched, a.model, device, scaler, True)
        with torch.no_grad():
            vl = run_epoch(model, dv, None, None, a.model, device, scaler, False)
        log.append({'epoch': ep + 1, 'train_loss': round(tl, 4), 'val_loss': round(vl, 4)})
        if vl < best:
            best = vl
            torch.save(model.state_dict(), run / 'best.pt')
        print(f'epoch {ep + 1}/{a.epochs} train {tl:.4f} val {vl:.4f}', flush=True)
    train_min = (time.time() - t0) / 60
    model.load_state_dict(torch.load(run / 'best.pt', map_location=device))
    preds, ms = predict(model, te, cls, a.model, a, device)
    vpred, _ = predict(model, va, cls, a.model, a, device)
    r = {'run': a.name, 'kind': a.model, 'note': a.note,
         'settings': {'epochs': a.epochs, 'batch': a.batch, 'lr': lr, 'input': [a.H, a.W], 'stack_two_lines': a.stack,
                      'flip': a.flip, 'train_rows': len(tr), 'real_rows': n_real, 'extra': a.train_extra,
                      'optimizer': 'AdamW + OneCycle', 'seed': a.seed},
         'train_minutes': round(train_min, 1), 'params_M': round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
         'val_ocr': metrics.score_rows((p['pred'], p['true']) for p in vpred),
         'ocr_test': {'rows_decoder': metrics.score_rows((p['pred'], p['true']) for p in preds),
                      'ms_per_plate': round(ms, 2),
                      'by_source': {s: metrics.score_rows((p['pred'], p['true']) for p in preds if p['source'] == s)
                                    for s in sorted({p['source'] for p in preds})}},
         'log': log, 'env': common.env_info()}
    root = common.find_dataset()
    r['ocr_test']['by_condition'] = metrics.condition_scores(preds, root)
    r['ocr_test']['extra_tests'] = {}
    for sp in metrics.extra_splits(root):            # real fog crops, synthetic rain copies
        ep, _ = predict(model, load_rows(sp)[:a.max_test], cls, a.model, a, device)
        r['ocr_test']['extra_tests'][sp] = metrics.score_rows((p['pred'], p['true']) for p in ep)
    common.save_json(r, run / 'results.json')
    for name, rows in (('test', preds), ('val', vpred)):
        with open(run / f'ocr_preds_{name}.csv', 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader(); w.writerows(rows)
    torch.save({'state_dict': model.state_dict(), 'kind': a.model, 'classes': cls, 'H': a.H, 'W': a.W,
                'stack': a.stack}, run / 'model_full.pt')
    print('DONE', a.name, r['ocr_test']['rows_decoder'], f'{train_min:.1f} min')


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /kaggle/working/lprkit/ved.py
"""Transformer OCR: Hugging Face VisionEncoderDecoder (image encoder + text decoder) on plate crops (ocr/{split}.csv).

  python ved.py --encoder google/vit-base-patch16-384 --decoder sagorsarker/bangla-bert-base --name vit_bbert ...
  python ved.py --encoder google/vit-base-patch16-384 --decoder xlm-roberta-base --name vit_xlmr ...
  python ved.py --trocr microsoft/trocr-base-stage1 --name trocr ...

Paper #1 settings (Table I): dropout 0.3 (encoder, decoder, attention), max length 20, beam 3, no-repeat n-gram 0,
length penalty 1.0, weight decay 0.01, fp16, batch 6. Paper #32: random flip + rotation ±5°, AdamW (flip is an option
here only to measure its effect). Images are padded to a square with grey (default) or stretched (--stretch).
Writes results.json + ocr_preds_test.csv and ocr_preds_val.csv (for fusion, paper #24).
"""
import argparse
import csv
import os
import random
import sys
import time
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageOps

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import common
import metrics
import vocab
from seq import load_rows


class DS(torch.utils.data.Dataset):
    def __init__(self, rows, processor, tokenizer, max_len, train, stretch, flip):
        self.rows, self.proc, self.tok, self.max_len, self.train, self.stretch, self.flip = rows, processor, tokenizer, max_len, train, stretch, flip

    def __len__(self):
        return len(self.rows)

    def image(self, path):
        im = Image.open(path).convert('RGB')
        if self.train:
            im = im.rotate(random.uniform(-5, 5), resample=Image.BILINEAR, expand=False, fillcolor=(127, 127, 127))
            if self.flip and random.random() < 0.5:
                im = ImageOps.mirror(im)
        if not self.stretch:
            w, h = im.size
            s = max(w, h)
            canvas = Image.new('RGB', (s, s), (127, 127, 127))
            canvas.paste(im, ((s - w) // 2, (s - h) // 2))
            im = canvas
        return im

    def __getitem__(self, i):
        r = self.rows[i]
        px = self.proc(images=self.image(r['path']), return_tensors='pt').pixel_values[0]
        text = vocab.N(r['text'])
        ids = self.tok(text, padding='max_length', max_length=self.max_len, truncation=True).input_ids
        labels = torch.tensor([t if t != self.tok.pad_token_id else -100 for t in ids])
        return px, labels, i


def build(a):
    from transformers import (AutoImageProcessor, AutoTokenizer, TrOCRProcessor, VisionEncoderDecoderModel)
    if a.trocr:
        proc = TrOCRProcessor.from_pretrained(a.trocr)
        model = VisionEncoderDecoderModel.from_pretrained(a.trocr)
        image_proc, tok = proc.image_processor, proc.tokenizer
        model.config.decoder_start_token_id = model.config.decoder_start_token_id or tok.cls_token_id or tok.bos_token_id
    else:
        image_proc = AutoImageProcessor.from_pretrained(a.encoder)
        kw = {}
        if 'bert' in a.decoder.lower() and 'roberta' not in a.decoder.lower():
            kw = {'do_lower_case': False, 'strip_accents': False}   # BERT's accent stripping deletes Bangla vowel signs
        tok = AutoTokenizer.from_pretrained(a.decoder, **kw)
        model = VisionEncoderDecoderModel.from_encoder_decoder_pretrained(
            a.encoder, a.decoder,
            encoder_hidden_dropout_prob=a.dropout, encoder_attention_probs_dropout_prob=a.dropout,
            decoder_hidden_dropout_prob=a.dropout, decoder_attention_probs_dropout_prob=a.dropout)
        model.config.decoder.is_decoder = True
        model.config.decoder.add_cross_attention = True
        model.config.decoder_start_token_id = tok.cls_token_id if tok.cls_token_id is not None else tok.bos_token_id
        model.config.eos_token_id = tok.sep_token_id if tok.sep_token_id is not None else tok.eos_token_id
    model.config.pad_token_id = tok.pad_token_id
    model.config.vocab_size = model.config.decoder.vocab_size
    gc = model.generation_config
    gc.decoder_start_token_id = model.config.decoder_start_token_id
    gc.eos_token_id = model.config.eos_token_id
    gc.pad_token_id = tok.pad_token_id
    gc.max_length, gc.num_beams, gc.no_repeat_ngram_size, gc.length_penalty, gc.early_stopping = a.max_len, a.beams, 0, 1.0, True
    return model, image_proc, tok


@torch.no_grad()
def predict(model, tok, ds, device, bs=16):
    model.eval()
    dl = torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=False, num_workers=min(2, common.WORKERS))
    out = []
    for px, _, ids in dl:
        with torch.autocast('cuda', enabled=device.type == 'cuda'):
            gen = model.generate(px.to(device), output_scores=True, return_dict_in_generate=True)
        texts = tok.batch_decode(gen.sequences, skip_special_tokens=True)
        scores = gen.sequences_scores.exp().tolist() if getattr(gen, 'sequences_scores', None) is not None else [0.0] * len(texts)
        for i, t, s in zip(ids.tolist(), texts, scores):
            r = ds.rows[i]
            out.append({'file': r['file'], 'true': r['text'], 'pred': vocab.N(t.replace(' ##', '').strip()),
                        'conf': round(float(s), 4), 'source': r['source']})
    ms = []
    for i in range(min(60, len(ds))):
        px = ds[i][0][None].to(device)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.autocast('cuda', enabled=device.type == 'cuda'):
            model.generate(px)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        ms.append(1000 * (time.perf_counter() - t0))
    return out, float(np.mean(ms[10:])) if len(ms) > 10 else 0.0


def main():
    p = argparse.ArgumentParser()
    p.add_argument('--encoder', default='google/vit-base-patch16-384')
    p.add_argument('--decoder', default='sagorsarker/bangla-bert-base')
    p.add_argument('--trocr', default='', help='a full TrOCR checkpoint instead of encoder+decoder')
    p.add_argument('--name', required=True)
    p.add_argument('--project', required=True)
    p.add_argument('--epochs', type=int, default=10)
    p.add_argument('--batch', type=int, default=6)
    p.add_argument('--accum', type=int, default=1)
    p.add_argument('--lr', type=float, default=5e-5)
    p.add_argument('--wd', type=float, default=0.01)
    p.add_argument('--dropout', type=float, default=0.3)
    p.add_argument('--max-len', type=int, default=20)
    p.add_argument('--beams', type=int, default=3)
    p.add_argument('--stretch', action='store_true')
    p.add_argument('--flip', action='store_true')
    p.add_argument('--max-train', type=int, default=None)
    p.add_argument('--max-test', type=int, default=None)
    p.add_argument('--seed', type=int, default=42)
    p.add_argument('--note', default='')
    a = p.parse_args()
    random.seed(a.seed); np.random.seed(a.seed); torch.manual_seed(a.seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    run = Path(a.project) / a.name
    run.mkdir(parents=True, exist_ok=True)
    model, proc, tok = build(a)
    model.to(device)
    tr, va, te = load_rows('train')[:a.max_train], load_rows('val')[:a.max_test], load_rows('test')[:a.max_test]
    ds_tr = DS(tr, proc, tok, a.max_len, True, a.stretch, a.flip)
    dl = torch.utils.data.DataLoader(ds_tr, batch_size=a.batch, shuffle=True, num_workers=common.WORKERS, drop_last=True)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=a.wd)
    steps = a.epochs * len(dl) // a.accum
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / 300) * max(0.0, 1 - s / max(1, steps)))
    scaler = torch.amp.GradScaler(enabled=device.type == 'cuda')
    t0, log, step = time.time(), [], 0
    for ep in range(a.epochs):
        model.train()
        tot = n = 0
        for k, (px, labels, _) in enumerate(dl):
            with torch.autocast('cuda', enabled=device.type == 'cuda'):
                loss = model(pixel_values=px.to(device), labels=labels.to(device)).loss / a.accum
            scaler.scale(loss).backward()
            if (k + 1) % a.accum == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step(); step += 1
            tot += float(loss) * a.accum * px.shape[0]; n += px.shape[0]
        vpred, _ = predict(model, tok, DS(va[:300], proc, tok, a.max_len, False, a.stretch, False), device)
        vs = metrics.score_rows((x['pred'], x['true']) for x in vpred)
        log.append({'epoch': ep + 1, 'train_loss': round(tot / max(1, n), 4), 'val_CER_%': vs['CER_%'],
                    'val_exact_%': vs['plate_exact_%']})
        print(log[-1], flush=True)
        if not any(l['val_CER_%'] < vs['CER_%'] for l in log[:-1]):
            model.save_pretrained(run / 'best'); tok.save_pretrained(run / 'best')
    train_min = (time.time() - t0) / 60
    from transformers import VisionEncoderDecoderModel
    model = VisionEncoderDecoderModel.from_pretrained(run / 'best').to(device)
    preds, ms = predict(model, tok, DS(te, proc, tok, a.max_len, False, a.stretch, False), device)
    vpred, _ = predict(model, tok, DS(va, proc, tok, a.max_len, False, a.stretch, False), device)
    r = {'run': a.name, 'kind': 'vision-encoder-decoder', 'note': a.note,
         'settings': {'encoder': a.encoder if not a.trocr else a.trocr, 'decoder': a.decoder if not a.trocr else a.trocr,
                      'epochs': a.epochs, 'batch': a.batch * a.accum, 'lr': a.lr, 'weight_decay': a.wd,
                      'dropout': a.dropout, 'max_len': a.max_len, 'beams': a.beams, 'pad_to_square': not a.stretch,
                      'flip': a.flip, 'steps': step, 'train_rows': len(tr)},
         'train_minutes': round(train_min, 1), 'params_M': round(sum(p.numel() for p in model.parameters()) / 1e6, 1),
         'val_ocr': metrics.score_rows((x['pred'], x['true']) for x in vpred),
         'ocr_test': {'rows_decoder': metrics.score_rows((x['pred'], x['true']) for x in preds), 'ms_per_plate': round(ms, 1),
                      'by_source': {s: metrics.score_rows((x['pred'], x['true']) for x in preds if x['source'] == s)
                                    for s in sorted({x['source'] for x in preds})}},
         'log': log, 'env': common.env_info()}
    root = common.find_dataset()
    r['ocr_test']['by_condition'] = metrics.condition_scores(preds, root)
    r['ocr_test']['extra_tests'] = {}
    for sp in metrics.extra_splits(root):            # real fog crops, synthetic rain copies
        ep, _ = predict(model, tok, DS(load_rows(sp)[:a.max_test], proc, tok, a.max_len, False, a.stretch, False), device)
        r['ocr_test']['extra_tests'][sp] = metrics.score_rows((x['pred'], x['true']) for x in ep)
    common.save_json(r, run / 'results.json')
    for name, rows in (('test', preds), ('val', vpred)):
        with open(run / f'ocr_preds_{name}.csv', 'w', newline='', encoding='utf-8') as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader(); w.writerows(rows)
    print('DONE', a.name, r['ocr_test']['rows_decoder'], f'{train_min:.1f} min')


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /kaggle/working/lprkit/c36_sahi.py
"""Paper #36 (SAHI), original-paper check: the authors' OWN released predictions for every VisDrone row of Table 1,
re-scored with pycocotools against the official VisDrone2019-DET test-dev labels.

  python c36_sahi.py prep --root /kaggle/working/visdrone      (test-dev photos + labels, the authors' 20 result zips)
  python c36_sahi.py eval --root /kaggle/working/visdrone --name visdrone_rescore --project runs

Sources (pinned):
  labels   VisDrone2019-DET-test-dev.zip from the Ultralytics assets mirror (the official VisDrone release; 1,610 photos)
  results  github.com/fcakyon/small-object-detection-benchmark release v0.0.1 (the paper's footnote 1 repo, renamed
           from fcakyon/sahi-benchmark): <detector>_<setting>_visdrone_results.zip, each with result.json (COCO
           detections, conf 0.01) and eval.json (the authors' own score)
Ground truth = the authors' visdrone_to_coco.py: classes 1-10 (pedestrian ... motor) -> 0-9, 'ignored regions' (0)
and 'others' (11) dropped, boxes with zero area dropped. Image ids = c36_visdrone_map.ID_TO_FILE (their converter's
id order, recovered and verified, see that file). Metric = the paper's (p.4): COCO AP at IoU 0.5, all / small /
medium / large, max 500 detections.
xView (Table 2) is not checked: its data needs a registration at xviewdataset.org.
"""
import argparse
import contextlib
import io
import json
import os
import sys
import urllib.request
import zipfile
from pathlib import Path

import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import common
from c36_visdrone_map import ID_TO_FILE

LABELS = 'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-test-dev.zip'
RESULTS = 'https://github.com/fcakyon/small-object-detection-benchmark/releases/download/v0.0.1/{}_visdrone_results.zip'
# Table 1 (p.3): setting -> (AP50, AP50 small, AP50 medium, AP50 large)
TABLE1 = {
    'fcos_fi': (25.8, 14.2, 39.6, 45.1), 'fcos_sahi_po': (29.0, 18.9, 41.5, 46.4), 'fcos_sahi_fi_po': (31.0, 19.8, 44.6, 49.0),
    'fcos_sf_sahi_po': (38.1, 25.7, 54.8, 56.9), 'fcos_sf_sahi_fi_po': (38.5, 25.9, 55.4, 59.8),
    'vfnet_fi': (28.8, 16.8, 44.0, 47.5), 'vfnet_sahi_po': (32.0, 21.4, 45.8, 45.5), 'vfnet_sahi_fi_po': (33.9, 22.4, 49.1, 49.4),
    'vfnet_sf_sahi_po': (41.9, 29.7, 58.8, 60.6), 'vfnet_sf_sahi_fi_po': (42.2, 29.6, 59.2, 63.3),
    'tood_fi': (29.4, 18.1, 44.1, 50.0), 'tood_sahi': (31.9, 22.6, 44.0, 45.2), 'tood_sahi_po': (32.5, 22.8, 45.2, 43.6),
    'tood_sahi_fi': (34.6, 23.8, 48.5, 53.1), 'tood_sahi_fi_po': (34.7, 23.8, 48.9, 50.3), 'tood_sf_fi': (36.8, 24.4, 53.8, 66.4),
    'tood_sf_sahi': (42.5, 31.6, 58.0, 61.1), 'tood_sf_sahi_po': (43.1, 31.7, 59.0, 60.2), 'tood_sf_sahi_fi': (43.4, 31.7, 59.6, 65.6),
    'tood_sf_sahi_fi_po': (43.5, 31.7, 59.8, 65.4)}
ABSTRACT = {'fcos': (6.8, 12.7), 'vfnet': (5.1, 13.4), 'tood': (5.3, 14.5)}      # "SAHI +x AP", "with SF cumulative +y" (p.1)


def fetch(url, dst):
    if not dst.exists():
        print('download', url, flush=True)
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=300) as r, open(dst, 'wb') as f:
            while True:
                b = r.read(1 << 22)
                if not b:
                    break
                f.write(b)
    return dst


def prep(a):
    root = Path(a.root)
    root.mkdir(parents=True, exist_ok=True)
    if not (root / 'VisDrone2019-DET-test-dev' / 'annotations').exists():
        with zipfile.ZipFile(fetch(LABELS, root / 'testdev.zip')) as z:
            z.extractall(root)
        (root / 'testdev.zip').unlink()
    for s in (list(TABLE1) if not a.max else list(TABLE1)[:a.max]):
        d = root / 'results' / s
        if (d / 'result.json').exists():
            continue
        d.mkdir(parents=True, exist_ok=True)
        zp = fetch(RESULTS.format(s), root / f'{s}.zip')
        with zipfile.ZipFile(zp) as z:
            for n in ('result.json', 'eval.json'):
                (d / n).write_bytes(z.read(n))
        zp.unlink()
    print('test-dev photos:', len(list((root / 'VisDrone2019-DET-test-dev' / 'images').glob('*.jpg'))),
          '| result files:', len(list((root / 'results').glob('*/result.json'))))


def ground_truth(root):
    from PIL import Image
    from pycocotools.coco import COCO
    base = Path(root) / 'VisDrone2019-DET-test-dev'
    images, anns = [], []
    for iid, fn in sorted(ID_TO_FILE.items()):
        w, h = Image.open(base / 'images' / fn).size
        images.append({'id': iid, 'width': w, 'height': h, 'file_name': fn})
        for l in (base / 'annotations' / fn.replace('.jpg', '.txt')).read_text().splitlines():
            r = l.split(',')
            if len(r) > 5 and 1 <= int(r[5]) <= 10:
                bb = [int(r[0]), int(r[1]), int(r[2]), int(r[3])]
                if bb[2] * bb[3] > 0:
                    anns.append({'id': len(anns) + 1, 'image_id': iid, 'category_id': int(r[5]) - 1, 'bbox': bb,
                                 'area': bb[2] * bb[3], 'iscrowd': 0})
    gt = COCO()
    gt.dataset = {'images': images, 'annotations': anns, 'categories': [{'id': k, 'name': str(k)} for k in range(10)]}
    with contextlib.redirect_stdout(io.StringIO()):
        gt.createIndex()
    return gt


def ap50(gt, dets):
    from pycocotools.cocoeval import COCOeval
    with contextlib.redirect_stdout(io.StringIO()):
        dt = gt.loadRes(dets)
        e = COCOeval(gt, dt, 'bbox')
        e.params.maxDets = [1, 10, 500]
        e.evaluate(); e.accumulate()
    p = e.eval['precision'][0, :, :, :, -1]                     # IoU 0.5, recall points, classes, areas, maxDets 500
    return [round(100 * float(np.mean(p[:, :, k][p[:, :, k] > -1])), 1) for k in range(4)]


def evaluate(a):
    run = Path(a.project) / a.name
    run.mkdir(parents=True, exist_ok=True)
    gt = ground_truth(a.root)
    out = {'run': a.name, 'kind': 'sahi-check', 'photos': len(gt.imgs), 'boxes': len(gt.anns), 'settings': {}}
    for s, paper in TABLE1.items():
        d = Path(a.root) / 'results' / s
        if not (d / 'result.json').exists():
            continue
        res = json.loads((d / 'result.json').read_text())
        ev = json.loads((d / 'eval.json').read_text())
        ours = ap50(gt, res)
        own = [round(100 * ev[k], 1) for k in ('bbox_mAP50', 'bbox_mAP50_s', 'bbox_mAP50_m', 'bbox_mAP50_l')]
        out['settings'][s] = {'paper': dict(zip(('AP50', 'AP50_s', 'AP50_m', 'AP50_l'), paper)),
                              'authors_eval_json': dict(zip(('AP50', 'AP50_s', 'AP50_m', 'AP50_l'), own)),
                              'rescored': dict(zip(('AP50', 'AP50_s', 'AP50_m', 'AP50_l'), ours)),
                              'rescored_minus_paper_AP50_pp': round(ours[0] - paper[0], 1), 'detections': len(res)}
        print(s, 'paper', paper, '| their eval.json', own, '| re-scored', ours, flush=True)
    st = out['settings']
    gains = {}
    for det_, (sahi_claim, sf_claim) in ABSTRACT.items():
        fi = st.get(f'{det_}_fi')
        best_sahi = max((v['rescored']['AP50'] for k, v in st.items() if k.startswith(det_ + '_sahi')), default=None)
        best_all = max((v['rescored']['AP50'] for k, v in st.items() if k.startswith(det_ + '_')), default=None)
        if fi and best_sahi is not None:
            gains[det_] = {'abstract_sahi_gain': sahi_claim, 'table_sahi_gain': round(best_sahi - fi['rescored']['AP50'], 1),
                           'abstract_sf_gain': sf_claim, 'table_sf_gain': round(best_all - fi['rescored']['AP50'], 1)}
    out['abstract_vs_table'] = gains
    out['max_abs_diff_rescored_vs_paper_AP50'] = max((abs(v['rescored_minus_paper_AP50_pp']) for v in st.values()), default=None)
    out['env'] = common.env_info()
    common.save_json(out, run / 'results.json')
    print('abstract vs table:', json.dumps(gains))
    print('DONE', a.name)


if __name__ == '__main__':
    p = argparse.ArgumentParser()
    p.add_argument('mode', choices=['prep', 'eval'])
    p.add_argument('--root', default=os.path.join(common.WORK, 'visdrone'))
    p.add_argument('--max', type=int, default=None, help='prep: only the first N result files (quick test)')
    p.add_argument('--name', default='visdrone_rescore')
    p.add_argument('--project', default=os.path.join(common.WORK, 'runs'))
    a = p.parse_args()
    prep(a) if a.mode == 'prep' else evaluate(a)

In [ ]:
%%writefile /kaggle/working/lprkit/c36_visdrone_map.py
"""The image ids of the SAHI authors' released VisDrone2019-DET test-dev predictions (fcakyon/small-object-detection-
benchmark, release v0.0.1, result.json) -> file names. Their ids follow the unsorted directory order of their own
converter run, which is not published; this mapping was recovered on 27 Oct 2026 from box overlap (FCOS / TOOD /
VFNet full-image results, Hungarian assignment) and verified: with it, pycocotools reproduces the authors' own
eval.json exactly (AP50 / small / medium / large) for 4 result files, incl. one not used to build it."""
ID_TO_FILE = {
    1: '9999938_00000_d_0000209.jpg', 2: '9999952_00000_d_0000088.jpg', 3: '9999938_00000_d_0000302.jpg',
    4: '9999941_00000_d_0000010.jpg', 5: '9999938_00000_d_0000371.jpg', 6: '9999986_00000_d_0000033.jpg',
    7: '0000078_06777_d_0000020.jpg', 8: '9999973_00000_d_0000017.jpg', 9: '9999938_00000_d_0000398.jpg',
    10: '0000074_01218_d_0000002.jpg', 11: '9999952_00000_d_0000320.jpg', 12: '9999986_00000_d_0000058.jpg',
    13: '0000078_06363_d_0000017.jpg', 14: '9999952_00000_d_0000254.jpg', 15: '9999952_00000_d_0000281.jpg',
    16: '9999952_00000_d_0000031.jpg', 17: '9999973_00000_d_0000075.jpg', 18: '9999938_00000_d_0000342.jpg',
    19: '9999938_00000_d_0000322.jpg', 20: '9999952_00000_d_0000090.jpg', 21: '9999938_00000_d_0000289.jpg',
    22: '9999938_00000_d_0000296.jpg', 23: '9999979_00000_d_0000047.jpg', 24: '9999938_00000_d_0000157.jpg',
    25: '9999952_00000_d_0000330.jpg', 26: '9999952_00000_d_0000324.jpg', 27: '9999938_00000_d_0000274.jpg',
    28: '9999973_00000_d_0000127.jpg', 29: '9999938_00000_d_0000023.jpg', 30: '0000299_03500_d_0000104.jpg',
    31: '9999976_00000_d_0000007.jpg', 32: '9999963_00000_d_0000044.jpg', 33: '9999938_00000_d_0000229.jpg',
    34: '9999938_00000_d_0000184.jpg', 35: '0000299_05500_d_0000108.jpg', 36: '9999952_00000_d_0000204.jpg',
    37: '9999952_00000_d_0000292.jpg', 38: '9999938_00000_d_0000318.jpg', 39: '9999952_00000_d_0000142.jpg',
    40: '0000265_04500_d_0000010.jpg', 41: '9999938_00000_d_0000016.jpg', 42: '9999938_00000_d_0000351.jpg',
    43: '9999938_00000_d_0000337.jpg', 44: '9999973_00000_d_0000022.jpg', 45: '0000074_15298_d_0000030.jpg',
    46: '0000365_01500_d_0000226.jpg', 47: '9999993_00000_d_0000006.jpg', 48: '9999993_00000_d_0000018.jpg',
    49: '9999979_00000_d_0000027.jpg', 50: '9999973_00000_d_0000016.jpg', 51: '9999986_00000_d_0000028.jpg',
    52: '9999938_00000_d_0000163.jpg', 53: '9999976_00000_d_0000005.jpg', 54: '9999938_00000_d_0000386.jpg',
    55: '9999976_00000_d_0000025.jpg', 56: '9999952_00000_d_0000211.jpg', 57: '0000347_00800_d_0000049.jpg',
    58: '9999941_00000_d_0000024.jpg', 59: '9999963_00000_d_0000052.jpg', 60: '9999952_00000_d_0000075.jpg',
    61: '0000011_05068_d_0000008.jpg', 62: '9999993_00000_d_0000016.jpg', 63: '9999938_00000_d_0000117.jpg',
    64: '9999938_00000_d_0000482.jpg', 65: '9999979_00000_d_0000012.jpg', 66: '9999952_00000_d_0000110.jpg',
    67: '9999976_00000_d_0000019.jpg', 68: '0000074_11827_d_0000023.jpg', 69: '9999952_00000_d_0000028.jpg',
    70: '9999938_00000_d_0000255.jpg', 71: '9999976_00000_d_0000023.jpg', 72: '9999938_00000_d_0000026.jpg',
    73: '9999993_00000_d_0000024.jpg', 74: '9999938_00000_d_0000450.jpg', 75: '9999938_00000_d_0000092.jpg',
    76: '9999952_00000_d_0000199.jpg', 77: '9999938_00000_d_0000182.jpg', 78: '9999976_00000_d_0000022.jpg',
    79: '0000355_01000_d_0000208.jpg', 80: '9999986_00000_d_0000026.jpg', 81: '9999938_00000_d_0000039.jpg',
    82: '9999993_00000_d_0000022.jpg', 83: '0000320_04050_d_0000012.jpg', 84: '9999952_00000_d_0000137.jpg',
    85: '9999938_00000_d_0000295.jpg', 86: '9999952_00000_d_0000280.jpg', 87: '9999973_00000_d_0000131.jpg',
    88: '9999986_00000_d_0000051.jpg', 89: '9999938_00000_d_0000216.jpg', 90: '9999938_00000_d_0000500.jpg',
    91: '9999996_00000_d_0000021.jpg', 92: '9999952_00000_d_0000150.jpg', 93: '9999952_00000_d_0000268.jpg',
    94: '9999938_00000_d_0000268.jpg', 95: '9999938_00000_d_0000414.jpg', 96: '9999938_00000_d_0000363.jpg',
    97: '9999938_00000_d_0000185.jpg', 98: '9999963_00000_d_0000072.jpg', 99: '9999973_00000_d_0000109.jpg',
    100: '9999938_00000_d_0000476.jpg', 101: '9999938_00000_d_0000405.jpg', 102: '0000152_00100_d_0000145.jpg',
    103: '9999952_00000_d_0000063.jpg', 104: '9999952_00000_d_0000256.jpg', 105: '9999963_00000_d_0000056.jpg',
    106: '9999976_00000_d_0000013.jpg', 107: '9999973_00000_d_0000071.jpg', 108: '0000011_04202_d_0000007.jpg',
    109: '9999986_00000_d_0000006.jpg', 110: '9999938_00000_d_0000068.jpg', 111: '9999976_00000_d_0000012.jpg',
    112: '9999952_00000_d_0000174.jpg', 113: '0000074_11362_d_0000022.jpg', 114: '9999938_00000_d_0000004.jpg',
    115: '9999938_00000_d_0000076.jpg', 116: '9999952_00000_d_0000008.jpg', 117: '9999973_00000_d_0000002.jpg',
    118: '9999952_00000_d_0000055.jpg', 119: '9999952_00000_d_0000116.jpg', 120: '0000011_06000_d_0000009.jpg',
    121: '0000299_01800_d_0000100.jpg', 122: '9999938_00000_d_0000449.jpg', 123: '9999986_00000_d_0000049.jpg',
    124: '9999979_00000_d_0000031.jpg', 125: '9999938_00000_d_0000393.jpg', 126: '9999938_00000_d_0000494.jpg',
    127: '9999973_00000_d_0000124.jpg', 128: '9999952_00000_d_0000112.jpg', 129: '9999979_00000_d_0000019.jpg',
    130: '9999947_00000_d_0000027.jpg', 131: '9999952_00000_d_0000049.jpg', 132: '9999952_00000_d_0000155.jpg',
    133: '9999938_00000_d_0000191.jpg', 134: '9999938_00000_d_0000065.jpg', 135: '0000063_07000_d_0000008.jpg',
    136: '0000073_00377_d_0000001.jpg', 137: '0000353_05500_d_0000199.jpg', 138: '0000074_08202_d_0000016.jpg',
    139: '9999938_00000_d_0000105.jpg', 140: '9999993_00000_d_0000013.jpg', 141: '9999938_00000_d_0000375.jpg',
    142: '9999938_00000_d_0000060.jpg', 143: '0000347_00000_d_0000046.jpg', 144: '9999952_00000_d_0000062.jpg',
    145: '9999952_00000_d_0000159.jpg', 146: '9999986_00000_d_0000027.jpg', 147: '9999938_00000_d_0000417.jpg',
    148: '0000183_00031_d_0000176.jpg', 149: '9999938_00000_d_0000156.jpg', 150: '9999952_00000_d_0000082.jpg',
    151: '9999938_00000_d_0000058.jpg', 152: '9999938_00000_d_0000278.jpg', 153: '9999941_00000_d_0000016.jpg',
    154: '9999938_00000_d_0000310.jpg', 155: '9999947_00000_d_0000003.jpg', 156: '9999973_00000_d_0000061.jpg',
    157: '9999973_00000_d_0000096.jpg', 158: '9999952_00000_d_0000288.jpg', 159: '9999979_00000_d_0000017.jpg',
    160: '9999963_00000_d_0000015.jpg', 161: '9999938_00000_d_0000172.jpg', 162: '0000073_05999_d_0000007.jpg',
    163: '9999938_00000_d_0000338.jpg', 164: '9999952_00000_d_0000228.jpg', 165: '9999938_00000_d_0000433.jpg',
    166: '9999963_00000_d_0000003.jpg', 167: '0000078_04546_d_0000013.jpg', 168: '0000078_06506_d_0000018.jpg',
    169: '9999993_00000_d_0000020.jpg', 170: '9999938_00000_d_0000483.jpg', 171: '9999952_00000_d_0000070.jpg',
    172: '9999963_00000_d_0000060.jpg', 173: '9999952_00000_d_0000215.jpg', 174: '9999952_00000_d_0000293.jpg',
    175: '9999938_00000_d_0000138.jpg', 176: '9999941_00000_d_0000021.jpg', 177: '0000054_00786_d_0000001.jpg',
    178: '9999938_00000_d_0000213.jpg', 179: '0000314_00500_d_0000143.jpg', 180: '9999938_00000_d_0000081.jpg',
    181: '0000320_04200_d_0000013.jpg', 182: '9999938_00000_d_0000249.jpg', 183: '9999973_00000_d_0000038.jpg',
    184: '9999938_00000_d_0000341.jpg', 185: '0000321_00000_d_0000015.jpg', 186: '9999996_00000_d_0000017.jpg',
    187: '0000074_10730_d_0000021.jpg', 188: '9999944_00000_d_0000004.jpg', 189: '0000105_01556_d_0000069.jpg',
    190: '9999980_00000_d_0000004.jpg', 191: '9999938_00000_d_0000434.jpg', 192: '0000265_03500_d_0000008.jpg',
    193: '9999938_00000_d_0000011.jpg', 194: '9999996_00000_d_0000003.jpg', 195: '0000302_03866_d_0000117.jpg',
    196: '9999952_00000_d_0000238.jpg', 197: '9999973_00000_d_0000013.jpg', 198: '0000074_14850_d_0000029.jpg',
    199: '9999952_00000_d_0000177.jpg', 200: '9999993_00000_d_0000005.jpg', 201: '9999944_00000_d_0000003.jpg',
    202: '0000278_02351_d_0000002.jpg', 203: '9999973_00000_d_0000051.jpg', 204: '9999952_00000_d_0000222.jpg',
    205: '9999938_00000_d_0000177.jpg', 206: '9999979_00000_d_0000022.jpg', 207: '9999973_00000_d_0000047.jpg',
    208: '9999938_00000_d_0000196.jpg', 209: '9999952_00000_d_0000276.jpg', 210: '0000353_05000_d_0000198.jpg',
    211: '9999938_00000_d_0000003.jpg', 212: '9999952_00000_d_0000345.jpg', 213: '9999938_00000_d_0000478.jpg',
    214: '9999938_00000_d_0000501.jpg', 215: '9999952_00000_d_0000321.jpg', 216: '9999947_00000_d_0000005.jpg',
    217: '9999973_00000_d_0000005.jpg', 218: '9999938_00000_d_0000218.jpg', 219: '9999963_00000_d_0000043.jpg',
    220: '0000353_00000_d_0000188.jpg', 221: '9999938_00000_d_0000365.jpg', 222: '9999938_00000_d_0000070.jpg',
    223: '0000320_00450_d_0000005.jpg', 224: '9999973_00000_d_0000110.jpg', 225: '9999938_00000_d_0000127.jpg',
    226: '9999938_00000_d_0000194.jpg', 227: '9999973_00000_d_0000041.jpg', 228: '9999952_00000_d_0000100.jpg',
    229: '9999938_00000_d_0000212.jpg', 230: '9999973_00000_d_0000117.jpg', 231: '0000207_00300_d_0000004.jpg',
    232: '9999938_00000_d_0000205.jpg', 233: '9999938_00000_d_0000054.jpg', 234: '9999938_00000_d_0000106.jpg',
    235: '9999973_00000_d_0000094.jpg', 236: '9999973_00000_d_0000081.jpg', 237: '9999996_00000_d_0000020.jpg',
    238: '9999986_00000_d_0000023.jpg', 239: '0000164_00078_d_0000159.jpg', 240: '9999952_00000_d_0000052.jpg',
    241: '9999952_00000_d_0000115.jpg', 242: '9999941_00000_d_0000001.jpg', 243: '9999952_00000_d_0000296.jpg',
    244: '9999973_00000_d_0000107.jpg', 245: '9999938_00000_d_0000287.jpg', 246: '9999996_00000_d_0000025.jpg',
    247: '9999952_00000_d_0000061.jpg', 248: '9999963_00000_d_0000048.jpg', 249: '9999952_00000_d_0000094.jpg',
    250: '9999979_00000_d_0000039.jpg', 251: '9999938_00000_d_0000489.jpg', 252: '9999973_00000_d_0000151.jpg',
    253: '9999938_00000_d_0000440.jpg', 254: '0000006_04309_d_0000011.jpg', 255: '9999952_00000_d_0000096.jpg',
    256: '0000078_03171_d_0000009.jpg', 257: '9999938_00000_d_0000437.jpg', 258: '9999938_00000_d_0000233.jpg',
    259: '9999938_00000_d_0000484.jpg', 260: '9999938_00000_d_0000030.jpg', 261: '9999952_00000_d_0000229.jpg',
    262: '0000183_00216_d_0000177.jpg', 263: '0000314_01500_d_0000145.jpg', 264: '9999952_00000_d_0000133.jpg',
    265: '9999952_00000_d_0000139.jpg', 266: '9999952_00000_d_0000068.jpg', 267: '9999938_00000_d_0000056.jpg',
    268: '9999952_00000_d_0000251.jpg', 269: '0000337_00500_d_0000037.jpg', 270: '9999973_00000_d_0000122.jpg',
    271: '9999938_00000_d_0000146.jpg', 272: '0000189_00297_d_0000198.jpg', 273: '0000337_03500_d_0000043.jpg',
    274: '9999938_00000_d_0000050.jpg', 275: '9999976_00000_d_0000002.jpg', 276: '9999952_00000_d_0000344.jpg',
    277: '9999938_00000_d_0000275.jpg', 278: '9999952_00000_d_0000146.jpg', 279: '0000186_00901_d_0000187.jpg',
    280: '9999938_00000_d_0000497.jpg', 281: '9999963_00000_d_0000021.jpg', 282: '9999963_00000_d_0000065.jpg',
    283: '9999947_00000_d_0000022.jpg', 284: '9999979_00000_d_0000018.jpg', 285: '9999938_00000_d_0000422.jpg',
    286: '9999938_00000_d_0000364.jpg', 287: '9999952_00000_d_0000103.jpg', 288: '9999996_00000_d_0000015.jpg',
    289: '9999963_00000_d_0000063.jpg', 290: '9999952_00000_d_0000014.jpg', 291: '9999938_00000_d_0000423.jpg',
    292: '0000369_02000_d_0000245.jpg', 293: '9999938_00000_d_0000252.jpg', 294: '9999973_00000_d_0000111.jpg',
    295: '9999952_00000_d_0000327.jpg', 296: '9999938_00000_d_0000392.jpg', 297: '9999963_00000_d_0000032.jpg',
    298: '9999938_00000_d_0000385.jpg', 299: '9999938_00000_d_0000403.jpg', 300: '9999979_00000_d_0000050.jpg',
    301: '0000272_00000_d_0000001.jpg', 302: '9999979_00000_d_0000016.jpg', 303: '9999952_00000_d_0000025.jpg',
    304: '9999941_00000_d_0000018.jpg', 305: '9999952_00000_d_0000149.jpg', 306: '0000078_07266_d_0000021.jpg',
    307: '9999938_00000_d_0000491.jpg', 308: '9999938_00000_d_0000044.jpg', 309: '9999973_00000_d_0000093.jpg',
    310: '9999938_00000_d_0000498.jpg', 311: '9999938_00000_d_0000293.jpg', 312: '9999938_00000_d_0000175.jpg',
    313: '9999938_00000_d_0000069.jpg', 314: '9999938_00000_d_0000285.jpg', 315: '9999938_00000_d_0000113.jpg',
    316: '9999973_00000_d_0000085.jpg', 317: '9999952_00000_d_0000269.jpg', 318: '9999938_00000_d_0000273.jpg',
    319: '0000006_02138_d_0000006.jpg', 320: '9999952_00000_d_0000073.jpg', 321: '0000074_05715_d_0000011.jpg',
    322: '9999938_00000_d_0000114.jpg', 323: '9999938_00000_d_0000239.jpg', 324: '9999996_00000_d_0000027.jpg',
    325: '9999979_00000_d_0000010.jpg', 326: '9999938_00000_d_0000344.jpg', 327: '9999952_00000_d_0000253.jpg',
    328: '9999993_00000_d_0000003.jpg', 329: '9999952_00000_d_0000169.jpg', 330: '9999938_00000_d_0000082.jpg',
    331: '0000073_04009_d_0000005.jpg', 332: '9999979_00000_d_0000041.jpg', 333: '9999952_00000_d_0000277.jpg',
    334: '9999938_00000_d_0000397.jpg', 335: '9999952_00000_d_0000038.jpg', 336: '9999986_00000_d_0000034.jpg',
    337: '9999952_00000_d_0000172.jpg', 338: '0000006_03636_d_0000009.jpg', 339: '9999952_00000_d_0000084.jpg',
    340: '0000369_00500_d_0000242.jpg', 341: '0000337_01500_d_0000039.jpg', 342: '0000063_02235_d_0000003.jpg',
    343: '9999938_00000_d_0000508.jpg', 344: '9999938_00000_d_0000427.jpg', 345: '0000259_01000_d_0000003.jpg',
    346: '9999952_00000_d_0000328.jpg', 347: '9999952_00000_d_0000138.jpg', 348: '9999952_00000_d_0000323.jpg',
    349: '9999938_00000_d_0000230.jpg', 350: '9999938_00000_d_0000366.jpg', 351: '9999973_00000_d_0000139.jpg',
    352: '9999973_00000_d_0000082.jpg', 353: '9999952_00000_d_0000248.jpg', 354: '9999952_00000_d_0000104.jpg',
    355: '9999938_00000_d_0000446.jpg', 356: '9999963_00000_d_0000046.jpg', 357: '9999938_00000_d_0000361.jpg',
    358: '9999938_00000_d_0000103.jpg', 359: '9999952_00000_d_0000059.jpg', 360: '9999938_00000_d_0000071.jpg',
    361: '9999963_00000_d_0000037.jpg', 362: '9999938_00000_d_0000169.jpg', 363: '9999941_00000_d_0000011.jpg',
    364: '9999979_00000_d_0000053.jpg', 365: '9999973_00000_d_0000113.jpg', 366: '9999963_00000_d_0000054.jpg',
    367: '9999963_00000_d_0000028.jpg', 368: '9999952_00000_d_0000340.jpg', 369: '9999973_00000_d_0000018.jpg',
    370: '9999938_00000_d_0000502.jpg', 371: '9999938_00000_d_0000419.jpg', 372: '9999938_00000_d_0000219.jpg',
    373: '9999980_00000_d_0000002.jpg', 374: '9999963_00000_d_0000033.jpg', 375: '0000259_03000_d_0000007.jpg',
    376: '9999993_00000_d_0000015.jpg', 377: '9999996_00000_d_0000006.jpg', 378: '9999963_00000_d_0000042.jpg',
    379: '0000006_06773_d_0000018.jpg', 380: '9999986_00000_d_0000024.jpg', 381: '9999963_00000_d_0000034.jpg',
    382: '0000272_03500_d_0000008.jpg', 383: '9999938_00000_d_0000349.jpg', 384: '9999938_00000_d_0000480.jpg',
    385: '9999952_00000_d_0000119.jpg', 386: '9999952_00000_d_0000230.jpg', 387: '9999938_00000_d_0000282.jpg',
    388: '9999996_00000_d_0000026.jpg', 389: '9999938_00000_d_0000451.jpg', 390: '9999952_00000_d_0000257.jpg',
    391: '9999952_00000_d_0000171.jpg', 392: '9999938_00000_d_0000326.jpg', 393: '0000066_01097_d_0000002.jpg',
    394: '9999938_00000_d_0000031.jpg', 395: '9999938_00000_d_0000493.jpg', 396: '9999996_00000_d_0000012.jpg',
    397: '0000190_00172_d_0000202.jpg', 398: '0000272_04000_d_0000009.jpg', 399: '9999973_00000_d_0000059.jpg',
    400: '9999996_00000_d_0000019.jpg', 401: '9999973_00000_d_0000035.jpg', 402: '9999938_00000_d_0000226.jpg',
    403: '0000120_06848_d_0000106.jpg', 404: '9999952_00000_d_0000252.jpg', 405: '9999963_00000_d_0000029.jpg',
    406: '9999973_00000_d_0000086.jpg', 407: '9999973_00000_d_0000129.jpg', 408: '9999963_00000_d_0000074.jpg',
    409: '9999963_00000_d_0000002.jpg', 410: '9999973_00000_d_0000053.jpg', 411: '9999938_00000_d_0000144.jpg',
    412: '9999938_00000_d_0000290.jpg', 413: '9999986_00000_d_0000011.jpg', 414: '9999973_00000_d_0000072.jpg',
    415: '9999947_00000_d_0000026.jpg', 416: '9999938_00000_d_0000118.jpg', 417: '9999938_00000_d_0000028.jpg',
    418: '9999973_00000_d_0000012.jpg', 419: '9999952_00000_d_0000310.jpg', 420: '9999973_00000_d_0000118.jpg',
    421: '0000272_02500_d_0000006.jpg', 422: '9999986_00000_d_0000050.jpg', 423: '9999986_00000_d_0000022.jpg',
    424: '9999941_00000_d_0000004.jpg', 425: '0000187_00444_d_0000190.jpg', 426: '9999938_00000_d_0000149.jpg',
    427: '9999952_00000_d_0000153.jpg', 428: '9999973_00000_d_0000045.jpg', 429: '0000152_03226_d_0000153.jpg',
    430: '0000006_00611_d_0000002.jpg', 431: '9999963_00000_d_0000073.jpg', 432: '9999952_00000_d_0000017.jpg',
    433: '9999976_00000_d_0000030.jpg', 434: '0000120_02946_d_0000098.jpg', 435: '9999986_00000_d_0000008.jpg',
    436: '9999938_00000_d_0000235.jpg', 437: '9999952_00000_d_0000058.jpg', 438: '9999973_00000_d_0000098.jpg',
    439: '9999938_00000_d_0000201.jpg', 440: '9999973_00000_d_0000067.jpg', 441: '9999976_00000_d_0000003.jpg',
    442: '0000265_00500_d_0000002.jpg', 443: '9999976_00000_d_0000031.jpg', 444: '9999952_00000_d_0000086.jpg',
    445: '9999952_00000_d_0000098.jpg', 446: '9999947_00000_d_0000021.jpg', 447: '9999952_00000_d_0000193.jpg',
    448: '0000337_01000_d_0000038.jpg', 449: '9999979_00000_d_0000001.jpg', 450: '9999986_00000_d_0000040.jpg',
    451: '9999941_00000_d_0000014.jpg', 452: '9999938_00000_d_0000334.jpg', 453: '9999952_00000_d_0000180.jpg',
    454: '9999938_00000_d_0000368.jpg', 455: '9999941_00000_d_0000019.jpg', 456: '9999952_00000_d_0000237.jpg',
    457: '9999952_00000_d_0000335.jpg', 458: '9999963_00000_d_0000017.jpg', 459: '9999973_00000_d_0000049.jpg',
    460: '0000353_01500_d_0000191.jpg', 461: '0000299_04000_d_0000105.jpg', 462: '9999938_00000_d_0000485.jpg',
    463: '9999938_00000_d_0000033.jpg', 464: '9999986_00000_d_0000012.jpg', 465: '9999952_00000_d_0000266.jpg',
    466: '9999973_00000_d_0000149.jpg', 467: '0000365_00500_d_0000224.jpg', 468: '0000328_00300_d_0000020.jpg',
    469: '0000073_01275_d_0000002.jpg', 470: '0000272_04500_d_0000010.jpg', 471: '9999938_00000_d_0000093.jpg',
    472: '9999938_00000_d_0000505.jpg', 473: '9999993_00000_d_0000014.jpg', 474: '9999952_00000_d_0000274.jpg',
    475: '9999952_00000_d_0000298.jpg', 476: '9999963_00000_d_0000020.jpg', 477: '9999952_00000_d_0000265.jpg',
    478: '9999952_00000_d_0000233.jpg', 479: '0000320_00150_d_0000003.jpg', 480: '9999938_00000_d_0000315.jpg',
    481: '0000161_01584_d_0000158.jpg', 482: '9999952_00000_d_0000047.jpg', 483: '0000314_01908_d_0000146.jpg',
    484: '9999938_00000_d_0000473.jpg', 485: '9999993_00000_d_0000021.jpg', 486: '9999938_00000_d_0000180.jpg',
    487: '9999973_00000_d_0000106.jpg', 488: '9999973_00000_d_0000105.jpg', 489: '9999952_00000_d_0000196.jpg',
    490: '9999938_00000_d_0000240.jpg', 491: '9999952_00000_d_0000156.jpg', 492: '9999938_00000_d_0000183.jpg',
    493: '9999952_00000_d_0000312.jpg', 494: '9999976_00000_d_0000010.jpg', 495: '9999952_00000_d_0000326.jpg',
    496: '9999952_00000_d_0000039.jpg', 497: '9999952_00000_d_0000158.jpg', 498: '9999938_00000_d_0000394.jpg',
    499: '9999938_00000_d_0000513.jpg', 500: '9999938_00000_d_0000220.jpg', 501: '0000074_02723_d_0000005.jpg',
    502: '9999938_00000_d_0000009.jpg', 503: '9999938_00000_d_0000207.jpg', 504: '9999973_00000_d_0000031.jpg',
    505: '9999941_00000_d_0000035.jpg', 506: '9999976_00000_d_0000029.jpg', 507: '0000259_00000_d_0000001.jpg',
    508: '9999938_00000_d_0000257.jpg', 509: '9999973_00000_d_0000116.jpg', 510: '0000063_03139_d_0000004.jpg',
    511: '9999938_00000_d_0000492.jpg', 512: '9999938_00000_d_0000458.jpg', 513: '0000074_13313_d_0000026.jpg',
    514: '9999941_00000_d_0000030.jpg', 515: '0000063_08995_d_0000010.jpg', 516: '9999993_00000_d_0000004.jpg',
    517: '9999938_00000_d_0000263.jpg', 518: '9999963_00000_d_0000045.jpg', 519: '9999996_00000_d_0000010.jpg',
    520: '9999938_00000_d_0000083.jpg', 521: '9999973_00000_d_0000007.jpg', 522: '9999952_00000_d_0000279.jpg',
    523: '9999938_00000_d_0000057.jpg', 524: '9999952_00000_d_0000010.jpg', 525: '9999938_00000_d_0000223.jpg',
    526: '9999938_00000_d_0000314.jpg', 527: '9999993_00000_d_0000009.jpg', 528: '9999996_00000_d_0000011.jpg',
    529: '9999938_00000_d_0000166.jpg', 530: '9999952_00000_d_0000226.jpg', 531: '9999938_00000_d_0000317.jpg',
    532: '9999996_00000_d_0000008.jpg', 533: '9999986_00000_d_0000038.jpg', 534: '9999952_00000_d_0000255.jpg',
    535: '9999938_00000_d_0000462.jpg', 536: '9999952_00000_d_0000290.jpg', 537: '9999973_00000_d_0000143.jpg',
    538: '9999938_00000_d_0000231.jpg', 539: '9999986_00000_d_0000004.jpg', 540: '9999947_00000_d_0000013.jpg',
    541: '9999938_00000_d_0000421.jpg', 542: '9999952_00000_d_0000173.jpg', 543: '0000087_00009_d_0000001.jpg',
    544: '9999963_00000_d_0000026.jpg', 545: '9999952_00000_d_0000182.jpg', 546: '9999938_00000_d_0000316.jpg',
    547: '9999952_00000_d_0000309.jpg', 548: '9999938_00000_d_0000448.jpg', 549: '9999952_00000_d_0000243.jpg',
    550: '9999938_00000_d_0000042.jpg', 551: '0000299_01650_d_0000099.jpg', 552: '9999952_00000_d_0000033.jpg',
    553: '9999952_00000_d_0000092.jpg', 554: '0000365_00000_d_0000223.jpg', 555: '9999947_00000_d_0000014.jpg',
    556: '9999979_00000_d_0000026.jpg', 557: '0000353_04500_d_0000197.jpg', 558: '9999952_00000_d_0000026.jpg',
    559: '9999938_00000_d_0000224.jpg', 560: '9999963_00000_d_0000035.jpg', 561: '9999952_00000_d_0000267.jpg',
    562: '9999938_00000_d_0000447.jpg', 563: '9999973_00000_d_0000128.jpg', 564: '9999963_00000_d_0000012.jpg',
    565: '9999938_00000_d_0000097.jpg', 566: '9999938_00000_d_0000514.jpg', 567: '0000321_00268_d_0000016.jpg',
    568: '9999938_00000_d_0000112.jpg', 569: '9999976_00000_d_0000033.jpg', 570: '9999973_00000_d_0000125.jpg',
    571: '9999938_00000_d_0000372.jpg', 572: '9999952_00000_d_0000117.jpg', 573: '9999938_00000_d_0000445.jpg',
    574: '9999952_00000_d_0000210.jpg', 575: '0000074_07850_d_0000015.jpg', 576: '9999973_00000_d_0000136.jpg',
    577: '9999938_00000_d_0000032.jpg', 578: '9999938_00000_d_0000355.jpg', 579: '9999938_00000_d_0000321.jpg',
    580: '9999963_00000_d_0000076.jpg', 581: '9999973_00000_d_0000063.jpg', 582: '9999938_00000_d_0000010.jpg',
    583: '9999938_00000_d_0000259.jpg', 584: '9999973_00000_d_0000054.jpg', 585: '9999938_00000_d_0000327.jpg',
    586: '0000207_00700_d_0000008.jpg', 587: '9999986_00000_d_0000057.jpg', 588: '9999938_00000_d_0000416.jpg',
    589: '9999986_00000_d_0000021.jpg', 590: '0000074_03738_d_0000007.jpg', 591: '9999952_00000_d_0000128.jpg',
    592: '9999952_00000_d_0000170.jpg', 593: '9999938_00000_d_0000012.jpg', 594: '9999938_00000_d_0000129.jpg',
    595: '9999938_00000_d_0000202.jpg', 596: '9999952_00000_d_0000188.jpg', 597: '9999952_00000_d_0000303.jpg',
    598: '9999952_00000_d_0000099.jpg', 599: '9999952_00000_d_0000305.jpg', 600: '9999938_00000_d_0000237.jpg',
    601: '9999973_00000_d_0000052.jpg', 602: '9999938_00000_d_0000040.jpg', 603: '9999976_00000_d_0000027.jpg',
    604: '9999941_00000_d_0000027.jpg', 605: '9999986_00000_d_0000029.jpg', 606: '9999938_00000_d_0000062.jpg',
    607: '9999938_00000_d_0000227.jpg', 608: '0000337_02500_d_0000041.jpg', 609: '9999938_00000_d_0000137.jpg',
    610: '9999952_00000_d_0000341.jpg', 611: '9999952_00000_d_0000270.jpg', 612: '9999938_00000_d_0000460.jpg',
    613: '9999947_00000_d_0000002.jpg', 614: '9999938_00000_d_0000085.jpg', 615: '9999976_00000_d_0000016.jpg',
    616: '9999952_00000_d_0000148.jpg', 617: '9999938_00000_d_0000108.jpg', 618: '9999952_00000_d_0000056.jpg',
    619: '9999986_00000_d_0000031.jpg', 620: '9999938_00000_d_0000053.jpg', 621: '9999938_00000_d_0000094.jpg',
    622: '9999952_00000_d_0000019.jpg', 623: '0000320_00750_d_0000007.jpg', 624: '9999938_00000_d_0000475.jpg',
    625: '0000298_00000_d_0000070.jpg', 626: '9999952_00000_d_0000249.jpg', 627: '9999938_00000_d_0000088.jpg',
    628: '0000187_00666_d_0000191.jpg', 629: '9999952_00000_d_0000295.jpg', 630: '0000299_05000_d_0000107.jpg',
    631: '9999938_00000_d_0000457.jpg', 632: '0000320_04375_d_0000014.jpg', 633: '9999979_00000_d_0000003.jpg',
    634: '9999952_00000_d_0000191.jpg', 635: '9999938_00000_d_0000079.jpg', 636: '9999993_00000_d_0000007.jpg',
    637: '9999938_00000_d_0000067.jpg', 638: '9999952_00000_d_0000317.jpg', 639: '9999973_00000_d_0000037.jpg',
    640: '9999938_00000_d_0000373.jpg', 641: '9999938_00000_d_0000154.jpg', 642: '9999952_00000_d_0000200.jpg',
    643: '9999986_00000_d_0000010.jpg', 644: '9999938_00000_d_0000253.jpg', 645: '9999938_00000_d_0000110.jpg',
    646: '9999973_00000_d_0000138.jpg', 647: '9999938_00000_d_0000066.jpg', 648: '0000207_00100_d_0000002.jpg',
    649: '9999973_00000_d_0000141.jpg', 650: '9999952_00000_d_0000337.jpg', 651: '9999938_00000_d_0000284.jpg',
    652: '9999986_00000_d_0000039.jpg', 653: '9999938_00000_d_0000324.jpg', 654: '9999986_00000_d_0000001.jpg',
    655: '9999952_00000_d_0000289.jpg', 656: '9999947_00000_d_0000023.jpg', 657: '9999952_00000_d_0000217.jpg',
    658: '9999952_00000_d_0000175.jpg', 659: '0000118_02632_d_0000077.jpg', 660: '0000073_03155_d_0000004.jpg',
    661: '9999973_00000_d_0000073.jpg', 662: '9999973_00000_d_0000039.jpg', 663: '9999963_00000_d_0000066.jpg',
    664: '9999947_00000_d_0000020.jpg', 665: '9999973_00000_d_0000056.jpg', 666: '9999976_00000_d_0000001.jpg',
    667: '9999938_00000_d_0000124.jpg', 668: '9999938_00000_d_0000488.jpg', 669: '9999963_00000_d_0000013.jpg',
    670: '9999938_00000_d_0000429.jpg', 671: '9999952_00000_d_0000132.jpg', 672: '9999952_00000_d_0000074.jpg',
    673: '9999938_00000_d_0000487.jpg', 674: '9999947_00000_d_0000016.jpg', 675: '9999973_00000_d_0000142.jpg',
    676: '9999979_00000_d_0000042.jpg', 677: '9999973_00000_d_0000112.jpg', 678: '9999973_00000_d_0000044.jpg',
    679: '9999938_00000_d_0000048.jpg', 680: '9999938_00000_d_0000101.jpg', 681: '9999963_00000_d_0000051.jpg',
    682: '9999986_00000_d_0000003.jpg', 683: '9999938_00000_d_0000435.jpg', 684: '9999952_00000_d_0000283.jpg',
    685: '9999938_00000_d_0000383.jpg', 686: '9999973_00000_d_0000010.jpg', 687: '9999938_00000_d_0000073.jpg',
    688: '9999973_00000_d_0000008.jpg', 689: '0000074_09738_d_0000019.jpg', 690: '9999952_00000_d_0000004.jpg',
    691: '0000272_01500_d_0000004.jpg', 692: '9999938_00000_d_0000301.jpg', 693: '9999973_00000_d_0000115.jpg',
    694: '9999938_00000_d_0000379.jpg', 695: '9999973_00000_d_0000042.jpg', 696: '9999938_00000_d_0000283.jpg',
    697: '9999986_00000_d_0000037.jpg', 698: '9999938_00000_d_0000198.jpg', 699: '0000265_04000_d_0000009.jpg',
    700: '9999938_00000_d_0000424.jpg', 701: '9999952_00000_d_0000236.jpg', 702: '9999938_00000_d_0000247.jpg',
    703: '9999947_00000_d_0000006.jpg', 704: '9999952_00000_d_0000131.jpg', 705: '9999952_00000_d_0000087.jpg',
    706: '9999952_00000_d_0000135.jpg', 707: '9999973_00000_d_0000099.jpg', 708: '9999973_00000_d_0000032.jpg',
    709: '9999952_00000_d_0000080.jpg', 710: '9999938_00000_d_0000415.jpg', 711: '9999938_00000_d_0000454.jpg',
    712: '9999952_00000_d_0000313.jpg', 713: '9999973_00000_d_0000066.jpg', 714: '9999963_00000_d_0000031.jpg',
    715: '9999952_00000_d_0000043.jpg', 716: '0000087_01580_d_0000005.jpg', 717: '9999938_00000_d_0000311.jpg',
    718: '9999963_00000_d_0000069.jpg', 719: '9999952_00000_d_0000261.jpg', 720: '9999938_00000_d_0000025.jpg',
    721: '0000184_00512_d_0000182.jpg', 722: '9999938_00000_d_0000453.jpg', 723: '9999938_00000_d_0000243.jpg',
    724: '9999976_00000_d_0000017.jpg', 725: '9999938_00000_d_0000059.jpg', 726: '9999947_00000_d_0000009.jpg',
    727: '9999952_00000_d_0000165.jpg', 728: '9999938_00000_d_0000074.jpg', 729: '9999952_00000_d_0000315.jpg',
    730: '0000006_01111_d_0000003.jpg', 731: '9999938_00000_d_0000333.jpg', 732: '9999938_00000_d_0000320.jpg',
    733: '9999938_00000_d_0000431.jpg', 734: '9999986_00000_d_0000052.jpg', 735: '9999973_00000_d_0000070.jpg',
    736: '9999952_00000_d_0000333.jpg', 737: '9999986_00000_d_0000053.jpg', 738: '9999938_00000_d_0000142.jpg',
    739: '9999938_00000_d_0000017.jpg', 740: '9999941_00000_d_0000009.jpg', 741: '9999973_00000_d_0000088.jpg',
    742: '0000120_00559_d_0000092.jpg', 743: '0000011_01350_d_0000003.jpg', 744: '0000152_02214_d_0000150.jpg',
    745: '0000347_01250_d_0000050.jpg', 746: '0000314_01000_d_0000144.jpg', 747: '9999986_00000_d_0000015.jpg',
    748: '9999952_00000_d_0000009.jpg', 749: '9999938_00000_d_0000425.jpg', 750: '9999938_00000_d_0000246.jpg',
    751: '9999938_00000_d_0000234.jpg', 752: '9999976_00000_d_0000021.jpg', 753: '9999993_00000_d_0000010.jpg',
    754: '0000063_12500_d_0000014.jpg', 755: '0000370_03000_d_0000256.jpg', 756: '0000353_00500_d_0000189.jpg',
    757: '9999938_00000_d_0000388.jpg', 758: '9999941_00000_d_0000025.jpg', 759: '0000006_05168_d_0000013.jpg',
    760: '9999963_00000_d_0000008.jpg', 761: '0000063_08000_d_0000009.jpg', 762: '9999952_00000_d_0000262.jpg',
    763: '9999952_00000_d_0000044.jpg', 764: '0000105_01748_d_0000070.jpg', 765: '9999976_00000_d_0000024.jpg',
    766: '9999973_00000_d_0000145.jpg', 767: '9999973_00000_d_0000147.jpg', 768: '9999952_00000_d_0000072.jpg',
    769: '9999938_00000_d_0000089.jpg', 770: '9999941_00000_d_0000023.jpg', 771: '9999973_00000_d_0000104.jpg',
    772: '9999952_00000_d_0000240.jpg', 773: '0000152_00599_d_0000146.jpg', 774: '9999938_00000_d_0000410.jpg',
    775: '9999963_00000_d_0000053.jpg', 776: '9999952_00000_d_0000187.jpg', 777: '0000310_05822_d_0000130.jpg',
    778: '9999973_00000_d_0000095.jpg', 779: '9999979_00000_d_0000011.jpg', 780: '9999952_00000_d_0000152.jpg',
    781: '0000320_01050_d_0000009.jpg', 782: '0000011_01955_d_0000004.jpg', 783: '9999952_00000_d_0000113.jpg',
    784: '9999986_00000_d_0000018.jpg', 785: '0000294_00000_d_0000053.jpg', 786: '9999952_00000_d_0000129.jpg',
    787: '9999952_00000_d_0000145.jpg', 788: '9999938_00000_d_0000369.jpg', 789: '9999938_00000_d_0000064.jpg',
    790: '9999938_00000_d_0000436.jpg', 791: '9999980_00000_d_0000003.jpg', 792: '9999996_00000_d_0000022.jpg',
    793: '9999941_00000_d_0000013.jpg', 794: '9999938_00000_d_0000336.jpg', 795: '9999986_00000_d_0000032.jpg',
    796: '9999938_00000_d_0000199.jpg', 797: '9999973_00000_d_0000027.jpg', 798: '9999952_00000_d_0000161.jpg',
    799: '9999938_00000_d_0000210.jpg', 800: '9999952_00000_d_0000029.jpg', 801: '9999941_00000_d_0000002.jpg',
    802: '9999963_00000_d_0000062.jpg', 803: '9999952_00000_d_0000189.jpg', 804: '9999963_00000_d_0000027.jpg',
    805: '0000063_04000_d_0000005.jpg', 806: '9999952_00000_d_0000242.jpg', 807: '0000078_00490_d_0000002.jpg',
    808: '9999952_00000_d_0000179.jpg', 809: '0000078_02778_d_0000008.jpg', 810: '9999938_00000_d_0000432.jpg',
    811: '9999941_00000_d_0000034.jpg', 812: '9999963_00000_d_0000040.jpg', 813: '9999938_00000_d_0000165.jpg',
    814: '9999938_00000_d_0000021.jpg', 815: '0000272_01000_d_0000003.jpg', 816: '0000320_03950_d_0000011.jpg',
    817: '9999980_00000_d_0000001.jpg', 818: '0000272_02000_d_0000005.jpg', 819: '0000278_02951_d_0000008.jpg',
    820: '9999963_00000_d_0000039.jpg', 821: '0000282_04000_d_0000009.jpg', 822: '9999952_00000_d_0000260.jpg',
    823: '9999952_00000_d_0000297.jpg', 824: '9999938_00000_d_0000104.jpg', 825: '9999941_00000_d_0000031.jpg',
    826: '9999938_00000_d_0000439.jpg', 827: '9999938_00000_d_0000102.jpg', 828: '9999952_00000_d_0000079.jpg',
    829: '9999996_00000_d_0000002.jpg', 830: '9999979_00000_d_0000028.jpg', 831: '9999963_00000_d_0000036.jpg',
    832: '9999976_00000_d_0000032.jpg', 833: '9999976_00000_d_0000020.jpg', 834: '9999963_00000_d_0000067.jpg',
    835: '9999947_00000_d_0000015.jpg', 836: '9999938_00000_d_0000037.jpg', 837: '9999952_00000_d_0000106.jpg',
    838: '9999938_00000_d_0000459.jpg', 839: '9999973_00000_d_0000020.jpg', 840: '9999963_00000_d_0000038.jpg',
    841: '9999952_00000_d_0000057.jpg', 842: '9999938_00000_d_0000509.jpg', 843: '9999938_00000_d_0000086.jpg',
    844: '9999963_00000_d_0000007.jpg', 845: '9999952_00000_d_0000054.jpg', 846: '9999938_00000_d_0000277.jpg',
    847: '9999952_00000_d_0000285.jpg', 848: '9999952_00000_d_0000003.jpg', 849: '0000164_01068_d_0000162.jpg',
    850: '9999938_00000_d_0000250.jpg', 851: '9999952_00000_d_0000291.jpg', 852: '9999938_00000_d_0000189.jpg',
    853: '0000337_04362_d_0000045.jpg', 854: '9999952_00000_d_0000085.jpg', 855: '9999952_00000_d_0000157.jpg',
    856: '0000074_15793_d_0000031.jpg', 857: '9999952_00000_d_0000301.jpg', 858: '9999941_00000_d_0000032.jpg',
    859: '9999938_00000_d_0000306.jpg', 860: '9999973_00000_d_0000119.jpg', 861: '9999952_00000_d_0000163.jpg',
    862: '9999971_00000_d_0000001.jpg', 863: '9999938_00000_d_0000221.jpg', 864: '9999952_00000_d_0000287.jpg',
    865: '9999952_00000_d_0000329.jpg', 866: '9999938_00000_d_0000339.jpg', 867: '9999938_00000_d_0000135.jpg',
    868: '9999979_00000_d_0000037.jpg', 869: '0000320_00600_d_0000006.jpg', 870: '9999938_00000_d_0000467.jpg',
    871: '9999973_00000_d_0000091.jpg', 872: '9999952_00000_d_0000164.jpg', 873: '9999938_00000_d_0000136.jpg',
    874: '9999938_00000_d_0000380.jpg', 875: '9999938_00000_d_0000150.jpg', 876: '9999979_00000_d_0000051.jpg',
    877: '9999952_00000_d_0000202.jpg', 878: '9999952_00000_d_0000126.jpg', 879: '9999938_00000_d_0000018.jpg',
    880: '0000299_00000_d_0000088.jpg', 881: '9999952_00000_d_0000136.jpg', 882: '9999938_00000_d_0000236.jpg',
    883: '0000074_12322_d_0000024.jpg', 884: '9999973_00000_d_0000069.jpg', 885: '9999941_00000_d_0000005.jpg',
    886: '9999938_00000_d_0000178.jpg', 887: '9999952_00000_d_0000143.jpg', 888: '9999993_00000_d_0000011.jpg',
    889: '9999938_00000_d_0000357.jpg', 890: '9999938_00000_d_0000313.jpg', 891: '9999938_00000_d_0000096.jpg',
    892: '9999938_00000_d_0000120.jpg', 893: '9999952_00000_d_0000013.jpg', 894: '9999938_00000_d_0000098.jpg',
    895: '9999973_00000_d_0000080.jpg', 896: '9999973_00000_d_0000102.jpg', 897: '9999938_00000_d_0000269.jpg',
    898: '0000190_00023_d_0000201.jpg', 899: '9999952_00000_d_0000332.jpg', 900: '9999986_00000_d_0000045.jpg',
    901: '9999963_00000_d_0000068.jpg', 902: '9999979_00000_d_0000014.jpg', 903: '9999986_00000_d_0000055.jpg',
    904: '9999952_00000_d_0000231.jpg', 905: '9999952_00000_d_0000120.jpg', 906: '9999941_00000_d_0000020.jpg',
    907: '9999938_00000_d_0000200.jpg', 908: '0000164_01840_d_0000165.jpg', 909: '9999979_00000_d_0000005.jpg',
    910: '9999938_00000_d_0000441.jpg', 911: '9999986_00000_d_0000035.jpg', 912: '9999952_00000_d_0000166.jpg',
    913: '9999938_00000_d_0000469.jpg', 914: '9999941_00000_d_0000026.jpg', 915: '9999979_00000_d_0000040.jpg',
    916: '9999996_00000_d_0000014.jpg', 917: '9999979_00000_d_0000025.jpg', 918: '9999938_00000_d_0000134.jpg',
    919: '9999973_00000_d_0000065.jpg', 920: '9999963_00000_d_0000004.jpg', 921: '0000259_02500_d_0000006.jpg',
    922: '9999938_00000_d_0000428.jpg', 923: '0000328_01050_d_0000025.jpg', 924: '0000078_01314_d_0000004.jpg',
    925: '9999938_00000_d_0000179.jpg', 926: '9999938_00000_d_0000258.jpg', 927: '9999973_00000_d_0000014.jpg',
    928: '9999938_00000_d_0000266.jpg', 929: '9999938_00000_d_0000279.jpg', 930: '9999973_00000_d_0000090.jpg',
    931: '9999938_00000_d_0000036.jpg', 932: '9999952_00000_d_0000130.jpg', 933: '9999938_00000_d_0000034.jpg',
    934: '9999952_00000_d_0000209.jpg', 935: '9999938_00000_d_0000468.jpg', 936: '9999973_00000_d_0000021.jpg',
    937: '9999963_00000_d_0000055.jpg', 938: '0000105_02747_d_0000073.jpg', 939: '9999938_00000_d_0000126.jpg',
    940: '9999938_00000_d_0000390.jpg', 941: '0000278_02251_d_0000001.jpg', 942: '9999973_00000_d_0000100.jpg',
    943: '9999952_00000_d_0000216.jpg', 944: '9999938_00000_d_0000328.jpg', 945: '9999952_00000_d_0000319.jpg',
    946: '9999938_00000_d_0000309.jpg', 947: '9999952_00000_d_0000336.jpg', 948: '9999952_00000_d_0000245.jpg',
    949: '9999938_00000_d_0000347.jpg', 950: '0000353_02000_d_0000192.jpg', 951: '9999979_00000_d_0000013.jpg',
    952: '9999941_00000_d_0000007.jpg', 953: '0000063_05000_d_0000006.jpg', 954: '0000006_05575_d_0000016.jpg',
    955: '0000074_08777_d_0000017.jpg', 956: '9999952_00000_d_0000037.jpg', 957: '0000314_00000_d_0000142.jpg',
    958: '9999952_00000_d_0000250.jpg', 959: '9999952_00000_d_0000102.jpg', 960: '9999938_00000_d_0000486.jpg',
    961: '0000063_00500_d_0000001.jpg', 962: '9999952_00000_d_0000294.jpg', 963: '9999952_00000_d_0000275.jpg',
    964: '9999963_00000_d_0000064.jpg', 965: '9999938_00000_d_0000384.jpg', 966: '9999973_00000_d_0000011.jpg',
    967: '9999952_00000_d_0000123.jpg', 968: '9999938_00000_d_0000348.jpg', 969: '9999947_00000_d_0000017.jpg',
    970: '9999952_00000_d_0000195.jpg', 971: '0000006_04050_d_0000010.jpg', 972: '9999963_00000_d_0000049.jpg',
    973: '9999941_00000_d_0000017.jpg', 974: '9999938_00000_d_0000014.jpg', 975: '9999938_00000_d_0000072.jpg',
    976: '9999938_00000_d_0000267.jpg', 977: '9999952_00000_d_0000212.jpg', 978: '9999938_00000_d_0000352.jpg',
    979: '9999938_00000_d_0000307.jpg', 980: '0000355_02500_d_0000211.jpg', 981: '9999979_00000_d_0000008.jpg',
    982: '0000328_00150_d_0000019.jpg', 983: '9999979_00000_d_0000004.jpg', 984: '9999938_00000_d_0000305.jpg',
    985: '0000078_05939_d_0000016.jpg', 986: '9999952_00000_d_0000192.jpg', 987: '9999973_00000_d_0000137.jpg',
    988: '9999938_00000_d_0000162.jpg', 989: '9999938_00000_d_0000387.jpg', 990: '9999952_00000_d_0000144.jpg',
    991: '9999938_00000_d_0000499.jpg', 992: '9999938_00000_d_0000299.jpg', 993: '9999938_00000_d_0000051.jpg',
    994: '9999938_00000_d_0000402.jpg', 995: '9999938_00000_d_0000147.jpg', 996: '9999952_00000_d_0000258.jpg',
    997: '9999952_00000_d_0000064.jpg', 998: '9999938_00000_d_0000330.jpg', 999: '9999980_00000_d_0000005.jpg',
    1000: '9999938_00000_d_0000251.jpg', 1001: '9999986_00000_d_0000043.jpg', 1002: '0000355_00000_d_0000206.jpg',
    1003: '9999973_00000_d_0000150.jpg', 1004: '9999938_00000_d_0000170.jpg', 1005: '9999938_00000_d_0000215.jpg',
    1006: '9999979_00000_d_0000029.jpg', 1007: '9999938_00000_d_0000504.jpg', 1008: '9999938_00000_d_0000358.jpg',
    1009: '9999938_00000_d_0000304.jpg', 1010: '9999952_00000_d_0000205.jpg', 1011: '9999976_00000_d_0000034.jpg',
    1012: '9999938_00000_d_0000238.jpg', 1013: '9999938_00000_d_0000168.jpg', 1014: '9999938_00000_d_0000389.jpg',
    1015: '9999952_00000_d_0000198.jpg', 1016: '9999938_00000_d_0000035.jpg', 1017: '9999938_00000_d_0000407.jpg',
    1018: '9999938_00000_d_0000276.jpg', 1019: '0000207_00000_d_0000001.jpg', 1020: '9999952_00000_d_0000220.jpg',
    1021: '9999941_00000_d_0000029.jpg', 1022: '9999973_00000_d_0000083.jpg', 1023: '0000278_02451_d_0000003.jpg',
    1024: '9999973_00000_d_0000120.jpg', 1025: '9999938_00000_d_0000008.jpg', 1026: '9999952_00000_d_0000227.jpg',
    1027: '9999973_00000_d_0000040.jpg', 1028: '9999938_00000_d_0000444.jpg', 1029: '9999938_00000_d_0000176.jpg',
    1030: '9999952_00000_d_0000030.jpg', 1031: '0000259_02000_d_0000005.jpg', 1032: '9999938_00000_d_0000516.jpg',
    1033: '9999938_00000_d_0000472.jpg', 1034: '9999938_00000_d_0000401.jpg', 1035: '9999952_00000_d_0000322.jpg',
    1036: '9999938_00000_d_0000265.jpg', 1037: '9999941_00000_d_0000015.jpg', 1038: '9999952_00000_d_0000334.jpg',
    1039: '9999952_00000_d_0000042.jpg', 1040: '9999941_00000_d_0000022.jpg', 1041: '9999941_00000_d_0000008.jpg',
    1042: '0000011_07307_d_0000011.jpg', 1043: '9999952_00000_d_0000024.jpg', 1044: '9999973_00000_d_0000148.jpg',
    1045: '9999938_00000_d_0000188.jpg', 1046: '9999973_00000_d_0000103.jpg', 1047: '9999952_00000_d_0000325.jpg',
    1048: '9999952_00000_d_0000147.jpg', 1049: '9999938_00000_d_0000161.jpg', 1050: '9999952_00000_d_0000206.jpg',
    1051: '9999938_00000_d_0000197.jpg', 1052: '9999938_00000_d_0000049.jpg', 1053: '9999944_00000_d_0000001.jpg',
    1054: '9999963_00000_d_0000023.jpg', 1055: '9999963_00000_d_0000010.jpg', 1056: '9999938_00000_d_0000077.jpg',
    1057: '9999952_00000_d_0000023.jpg', 1058: '9999963_00000_d_0000024.jpg', 1059: '9999938_00000_d_0000370.jpg',
    1060: '0000078_02363_d_0000007.jpg', 1061: '0000073_05010_d_0000006.jpg', 1062: '9999973_00000_d_0000077.jpg',
    1063: '9999938_00000_d_0000262.jpg', 1064: '9999986_00000_d_0000046.jpg', 1065: '9999938_00000_d_0000294.jpg',
    1066: '9999952_00000_d_0000060.jpg', 1067: '9999938_00000_d_0000111.jpg', 1068: '9999973_00000_d_0000003.jpg',
    1069: '9999993_00000_d_0000002.jpg', 1070: '9999938_00000_d_0000426.jpg', 1071: '9999938_00000_d_0000353.jpg',
    1072: '9999952_00000_d_0000032.jpg', 1073: '9999938_00000_d_0000465.jpg', 1074: '0000299_04500_d_0000106.jpg',
    1075: '9999938_00000_d_0000204.jpg', 1076: '9999947_00000_d_0000007.jpg', 1077: '9999993_00000_d_0000012.jpg',
    1078: '0000347_00511_d_0000048.jpg', 1079: '9999993_00000_d_0000001.jpg', 1080: '9999938_00000_d_0000019.jpg',
    1081: '9999938_00000_d_0000359.jpg', 1082: '9999952_00000_d_0000318.jpg', 1083: '9999938_00000_d_0000061.jpg',
    1084: '9999952_00000_d_0000207.jpg', 1085: '9999996_00000_d_0000004.jpg', 1086: '9999952_00000_d_0000076.jpg',
    1087: '9999963_00000_d_0000030.jpg', 1088: '9999938_00000_d_0000511.jpg', 1089: '9999938_00000_d_0000455.jpg',
    1090: '0000347_02000_d_0000052.jpg', 1091: '0000310_03000_d_0000124.jpg', 1092: '9999952_00000_d_0000114.jpg',
    1093: '0000120_05373_d_0000103.jpg', 1094: '0000011_02800_d_0000005.jpg', 1095: '9999976_00000_d_0000011.jpg',
    1096: '9999973_00000_d_0000026.jpg', 1097: '9999952_00000_d_0000201.jpg', 1098: '9999996_00000_d_0000001.jpg',
    1099: '9999938_00000_d_0000245.jpg', 1100: '9999979_00000_d_0000052.jpg', 1101: '9999973_00000_d_0000132.jpg',
    1102: '9999947_00000_d_0000004.jpg', 1103: '9999952_00000_d_0000083.jpg', 1104: '9999938_00000_d_0000020.jpg',
    1105: '9999938_00000_d_0000139.jpg', 1106: '0000353_06000_d_0000200.jpg', 1107: '9999938_00000_d_0000002.jpg',
    1108: '9999952_00000_d_0000122.jpg', 1109: '9999947_00000_d_0000011.jpg', 1110: '9999952_00000_d_0000089.jpg',
    1111: '0000299_02500_d_0000102.jpg', 1112: '9999938_00000_d_0000443.jpg', 1113: '9999973_00000_d_0000055.jpg',
    1114: '9999996_00000_d_0000028.jpg', 1115: '0000006_01275_d_0000004.jpg', 1116: '9999952_00000_d_0000168.jpg',
    1117: '9999938_00000_d_0000382.jpg', 1118: '9999952_00000_d_0000218.jpg', 1119: '9999979_00000_d_0000046.jpg',
    1120: '9999973_00000_d_0000130.jpg', 1121: '9999952_00000_d_0000223.jpg', 1122: '9999938_00000_d_0000107.jpg',
    1123: '0000164_01302_d_0000163.jpg', 1124: '9999979_00000_d_0000024.jpg', 1125: '0000353_03500_d_0000195.jpg',
    1126: '0000006_02616_d_0000007.jpg', 1127: '0000105_02931_d_0000074.jpg', 1128: '9999938_00000_d_0000271.jpg',
    1129: '9999979_00000_d_0000036.jpg', 1130: '9999938_00000_d_0000264.jpg', 1131: '9999979_00000_d_0000048.jpg',
    1132: '9999938_00000_d_0000470.jpg', 1133: '9999952_00000_d_0000221.jpg', 1134: '9999952_00000_d_0000111.jpg',
    1135: '9999938_00000_d_0000479.jpg', 1136: '9999938_00000_d_0000391.jpg', 1137: '9999952_00000_d_0000078.jpg',
    1138: '9999952_00000_d_0000272.jpg', 1139: '9999938_00000_d_0000029.jpg', 1140: '9999952_00000_d_0000311.jpg',
    1141: '9999938_00000_d_0000466.jpg', 1142: '9999986_00000_d_0000048.jpg', 1143: '9999938_00000_d_0000241.jpg',
    1144: '9999938_00000_d_0000046.jpg', 1145: '9999963_00000_d_0000070.jpg', 1146: '9999938_00000_d_0000343.jpg',
    1147: '9999952_00000_d_0000183.jpg', 1148: '9999973_00000_d_0000046.jpg', 1149: '0000272_05500_d_0000012.jpg',
    1150: '9999938_00000_d_0000160.jpg', 1151: '9999938_00000_d_0000319.jpg', 1152: '9999973_00000_d_0000133.jpg',
    1153: '9999973_00000_d_0000146.jpg', 1154: '9999986_00000_d_0000005.jpg', 1155: '0000011_06595_d_0000010.jpg',
    1156: '9999952_00000_d_0000095.jpg', 1157: '9999952_00000_d_0000127.jpg', 1158: '9999986_00000_d_0000030.jpg',
    1159: '9999979_00000_d_0000045.jpg', 1160: '9999947_00000_d_0000018.jpg', 1161: '0000105_02539_d_0000072.jpg',
    1162: '9999996_00000_d_0000023.jpg', 1163: '9999938_00000_d_0000297.jpg', 1164: '9999938_00000_d_0000158.jpg',
    1165: '9999976_00000_d_0000018.jpg', 1166: '9999973_00000_d_0000023.jpg', 1167: '9999938_00000_d_0000211.jpg',
    1168: '9999938_00000_d_0000140.jpg', 1169: '0000347_00100_d_0000047.jpg', 1170: '9999938_00000_d_0000132.jpg',
    1171: '9999938_00000_d_0000152.jpg', 1172: '9999952_00000_d_0000141.jpg', 1173: '9999952_00000_d_0000006.jpg',
    1174: '9999938_00000_d_0000181.jpg', 1175: '9999938_00000_d_0000099.jpg', 1176: '9999952_00000_d_0000259.jpg',
    1177: '9999973_00000_d_0000087.jpg', 1178: '9999938_00000_d_0000261.jpg', 1179: '9999976_00000_d_0000014.jpg',
    1180: '0000006_07596_d_0000020.jpg', 1181: '9999986_00000_d_0000007.jpg', 1182: '9999938_00000_d_0000442.jpg',
    1183: '9999938_00000_d_0000075.jpg', 1184: '9999947_00000_d_0000010.jpg', 1185: '9999938_00000_d_0000280.jpg',
    1186: '9999973_00000_d_0000009.jpg', 1187: '0000120_04920_d_0000102.jpg', 1188: '9999973_00000_d_0000036.jpg',
    1189: '0000337_02000_d_0000040.jpg', 1190: '9999952_00000_d_0000286.jpg', 1191: '9999938_00000_d_0000091.jpg',
    1192: '9999973_00000_d_0000076.jpg', 1193: '9999952_00000_d_0000020.jpg', 1194: '0000120_01535_d_0000095.jpg',
    1195: '9999979_00000_d_0000030.jpg', 1196: '9999952_00000_d_0000091.jpg', 1197: '0000083_00803_d_0000003.jpg',
    1198: '9999938_00000_d_0000515.jpg', 1199: '0000207_00500_d_0000006.jpg', 1200: '9999973_00000_d_0000029.jpg',
    1201: '9999952_00000_d_0000065.jpg', 1202: '9999973_00000_d_0000006.jpg', 1203: '9999938_00000_d_0000041.jpg',
    1204: '9999952_00000_d_0000300.jpg', 1205: '9999938_00000_d_0000354.jpg', 1206: '9999952_00000_d_0000176.jpg',
    1207: '9999963_00000_d_0000058.jpg', 1208: '9999973_00000_d_0000028.jpg', 1209: '9999952_00000_d_0000264.jpg',
    1210: '0000118_02327_d_0000076.jpg', 1211: '9999938_00000_d_0000173.jpg', 1212: '9999938_00000_d_0000406.jpg',
    1213: '9999963_00000_d_0000005.jpg', 1214: '9999952_00000_d_0000271.jpg', 1215: '0000078_01938_d_0000006.jpg',
    1216: '9999952_00000_d_0000203.jpg', 1217: '9999938_00000_d_0000456.jpg', 1218: '9999979_00000_d_0000033.jpg',
    1219: '9999938_00000_d_0000190.jpg', 1220: '9999973_00000_d_0000050.jpg', 1221: '9999938_00000_d_0000286.jpg',
    1222: '9999938_00000_d_0000335.jpg', 1223: '0000310_03500_d_0000125.jpg', 1224: '9999963_00000_d_0000059.jpg',
    1225: '9999952_00000_d_0000097.jpg', 1226: '9999973_00000_d_0000043.jpg', 1227: '9999938_00000_d_0000463.jpg',
    1228: '9999952_00000_d_0000012.jpg', 1229: '9999938_00000_d_0000005.jpg', 1230: '9999986_00000_d_0000013.jpg',
    1231: '9999938_00000_d_0000376.jpg', 1232: '9999952_00000_d_0000125.jpg', 1233: '0000207_00600_d_0000007.jpg',
    1234: '9999952_00000_d_0000077.jpg', 1235: '9999996_00000_d_0000016.jpg', 1236: '0000337_04000_d_0000044.jpg',
    1237: '9999938_00000_d_0000510.jpg', 1238: '9999941_00000_d_0000033.jpg', 1239: '9999952_00000_d_0000316.jpg',
    1240: '9999986_00000_d_0000002.jpg', 1241: '9999952_00000_d_0000167.jpg', 1242: '0000370_02000_d_0000254.jpg',
    1243: '0000063_11817_d_0000013.jpg', 1244: '9999963_00000_d_0000025.jpg', 1245: '9999938_00000_d_0000153.jpg',
    1246: '0000347_01728_d_0000051.jpg', 1247: '0000259_03500_d_0000008.jpg', 1248: '9999938_00000_d_0000125.jpg',
    1249: '9999938_00000_d_0000345.jpg', 1250: '9999952_00000_d_0000162.jpg', 1251: '0000353_01000_d_0000190.jpg',
    1252: '9999952_00000_d_0000045.jpg', 1253: '9999952_00000_d_0000005.jpg', 1254: '9999938_00000_d_0000203.jpg',
    1255: '9999938_00000_d_0000490.jpg', 1256: '0000337_03000_d_0000042.jpg', 1257: '9999952_00000_d_0000299.jpg',
    1258: '0000078_04210_d_0000012.jpg', 1259: '0000187_00024_d_0000189.jpg', 1260: '9999938_00000_d_0000143.jpg',
    1261: '9999973_00000_d_0000019.jpg', 1262: '9999973_00000_d_0000135.jpg', 1263: '0000353_02500_d_0000193.jpg',
    1264: '9999938_00000_d_0000133.jpg', 1265: '9999973_00000_d_0000114.jpg', 1266: '0000299_01200_d_0000096.jpg',
    1267: '9999963_00000_d_0000022.jpg', 1268: '9999938_00000_d_0000206.jpg', 1269: '9999993_00000_d_0000023.jpg',
    1270: '9999938_00000_d_0000115.jpg', 1271: '9999979_00000_d_0000006.jpg', 1272: '9999952_00000_d_0000051.jpg',
    1273: '9999941_00000_d_0000028.jpg', 1274: '9999938_00000_d_0000291.jpg', 1275: '9999952_00000_d_0000007.jpg',
    1276: '9999938_00000_d_0000260.jpg', 1277: '9999938_00000_d_0000141.jpg', 1278: '0000087_01140_d_0000004.jpg',
    1279: '9999952_00000_d_0000027.jpg', 1280: '9999952_00000_d_0000002.jpg', 1281: '0000011_00234_d_0000001.jpg',
    1282: '9999952_00000_d_0000307.jpg', 1283: '9999947_00000_d_0000001.jpg', 1284: '9999938_00000_d_0000503.jpg',
    1285: '9999996_00000_d_0000018.jpg', 1286: '9999973_00000_d_0000068.jpg', 1287: '9999952_00000_d_0000016.jpg',
    1288: '9999952_00000_d_0000331.jpg', 1289: '9999938_00000_d_0000187.jpg', 1290: '0000078_03315_d_0000010.jpg',
    1291: '0000250_02350_d_0000001.jpg', 1292: '9999952_00000_d_0000247.jpg', 1293: '9999938_00000_d_0000272.jpg',
    1294: '9999938_00000_d_0000171.jpg', 1295: '9999938_00000_d_0000024.jpg', 1296: '9999996_00000_d_0000009.jpg',
    1297: '9999976_00000_d_0000004.jpg', 1298: '9999938_00000_d_0000167.jpg', 1299: '9999947_00000_d_0000019.jpg',
    1300: '9999973_00000_d_0000058.jpg', 1301: '9999976_00000_d_0000015.jpg', 1302: '9999952_00000_d_0000244.jpg',
    1303: '9999952_00000_d_0000018.jpg', 1304: '9999973_00000_d_0000097.jpg', 1305: '0000006_00159_d_0000001.jpg',
    1306: '9999952_00000_d_0000194.jpg', 1307: '9999973_00000_d_0000144.jpg', 1308: '9999952_00000_d_0000181.jpg',
    1309: '0000078_06586_d_0000019.jpg', 1310: '9999938_00000_d_0000193.jpg', 1311: '9999938_00000_d_0000325.jpg',
    1312: '9999947_00000_d_0000012.jpg', 1313: '9999963_00000_d_0000006.jpg', 1314: '9999952_00000_d_0000234.jpg',
    1315: '9999952_00000_d_0000342.jpg', 1316: '9999938_00000_d_0000461.jpg', 1317: '9999938_00000_d_0000496.jpg',
    1318: '9999979_00000_d_0000043.jpg', 1319: '0000127_01002_d_0000123.jpg', 1320: '9999976_00000_d_0000008.jpg',
    1321: '9999986_00000_d_0000047.jpg', 1322: '9999973_00000_d_0000101.jpg', 1323: '9999952_00000_d_0000306.jpg',
    1324: '9999973_00000_d_0000123.jpg', 1325: '9999938_00000_d_0000332.jpg', 1326: '9999938_00000_d_0000100.jpg',
    1327: '9999952_00000_d_0000178.jpg', 1328: '9999952_00000_d_0000108.jpg', 1329: '9999938_00000_d_0000377.jpg',
    1330: '9999938_00000_d_0000288.jpg', 1331: '9999963_00000_d_0000001.jpg', 1332: '0000310_04000_d_0000126.jpg',
    1333: '0000320_01200_d_0000010.jpg', 1334: '0000120_07578_d_0000108.jpg', 1335: '9999952_00000_d_0000109.jpg',
    1336: '9999938_00000_d_0000329.jpg', 1337: '9999938_00000_d_0000095.jpg', 1338: '9999952_00000_d_0000338.jpg',
    1339: '0000127_00265_d_0000120.jpg', 1340: '9999952_00000_d_0000302.jpg', 1341: '9999952_00000_d_0000214.jpg',
    1342: '9999938_00000_d_0000242.jpg', 1343: '9999963_00000_d_0000016.jpg', 1344: '9999986_00000_d_0000042.jpg',
    1345: '9999938_00000_d_0000312.jpg', 1346: '9999938_00000_d_0000464.jpg', 1347: '9999952_00000_d_0000225.jpg',
    1348: '9999938_00000_d_0000222.jpg', 1349: '9999973_00000_d_0000030.jpg', 1350: '9999973_00000_d_0000134.jpg',
    1351: '9999952_00000_d_0000046.jpg', 1352: '9999938_00000_d_0000123.jpg', 1353: '9999952_00000_d_0000273.jpg',
    1354: '9999963_00000_d_0000071.jpg', 1355: '9999938_00000_d_0000109.jpg', 1356: '0000074_13875_d_0000027.jpg',
    1357: '9999938_00000_d_0000228.jpg', 1358: '9999952_00000_d_0000339.jpg', 1359: '9999952_00000_d_0000235.jpg',
    1360: '9999938_00000_d_0000248.jpg', 1361: '9999938_00000_d_0000244.jpg', 1362: '9999938_00000_d_0000164.jpg',
    1363: '0000074_07297_d_0000014.jpg', 1364: '9999963_00000_d_0000014.jpg', 1365: '9999986_00000_d_0000017.jpg',
    1366: '9999938_00000_d_0000396.jpg', 1367: '9999952_00000_d_0000015.jpg', 1368: '9999952_00000_d_0000151.jpg',
    1369: '9999952_00000_d_0000093.jpg', 1370: '9999973_00000_d_0000015.jpg', 1371: '9999938_00000_d_0000395.jpg',
    1372: '9999993_00000_d_0000008.jpg', 1373: '9999938_00000_d_0000013.jpg', 1374: '0000265_02000_d_0000005.jpg',
    1375: '0000120_01392_d_0000094.jpg', 1376: '9999938_00000_d_0000087.jpg', 1377: '9999952_00000_d_0000048.jpg',
    1378: '9999952_00000_d_0000053.jpg', 1379: '9999938_00000_d_0000340.jpg', 1380: '0000006_05999_d_0000017.jpg',
    1381: '9999938_00000_d_0000007.jpg', 1382: '9999938_00000_d_0000331.jpg', 1383: '9999938_00000_d_0000495.jpg',
    1384: '9999986_00000_d_0000044.jpg', 1385: '9999973_00000_d_0000079.jpg', 1386: '9999952_00000_d_0000308.jpg',
    1387: '9999952_00000_d_0000081.jpg', 1388: '9999979_00000_d_0000023.jpg', 1389: '9999938_00000_d_0000409.jpg',
    1390: '9999938_00000_d_0000399.jpg', 1391: '9999952_00000_d_0000304.jpg', 1392: '9999979_00000_d_0000049.jpg',
    1393: '0000265_01500_d_0000004.jpg', 1394: '9999938_00000_d_0000303.jpg', 1395: '9999952_00000_d_0000343.jpg',
    1396: '9999938_00000_d_0000151.jpg', 1397: '9999938_00000_d_0000043.jpg', 1398: '9999993_00000_d_0000017.jpg',
    1399: '9999952_00000_d_0000190.jpg', 1400: '9999973_00000_d_0000092.jpg', 1401: '9999963_00000_d_0000009.jpg',
    1402: '9999979_00000_d_0000035.jpg', 1403: '9999938_00000_d_0000052.jpg', 1404: '9999973_00000_d_0000074.jpg',
    1405: '9999938_00000_d_0000300.jpg', 1406: '9999952_00000_d_0000001.jpg', 1407: '9999986_00000_d_0000041.jpg',
    1408: '9999938_00000_d_0000001.jpg', 1409: '9999938_00000_d_0000481.jpg', 1410: '9999944_00000_d_0000002.jpg',
    1411: '9999938_00000_d_0000323.jpg', 1412: '9999963_00000_d_0000075.jpg', 1413: '9999973_00000_d_0000024.jpg',
    1414: '9999938_00000_d_0000411.jpg', 1415: '9999938_00000_d_0000195.jpg', 1416: '9999976_00000_d_0000009.jpg',
    1417: '9999979_00000_d_0000034.jpg', 1418: '9999979_00000_d_0000015.jpg', 1419: '0000074_10218_d_0000020.jpg',
    1420: '9999938_00000_d_0000507.jpg', 1421: '9999938_00000_d_0000292.jpg', 1422: '9999952_00000_d_0000107.jpg',
    1423: '9999938_00000_d_0000378.jpg', 1424: '9999996_00000_d_0000029.jpg', 1425: '9999938_00000_d_0000412.jpg',
    1426: '9999938_00000_d_0000308.jpg', 1427: '9999938_00000_d_0000078.jpg', 1428: '0000105_02201_d_0000071.jpg',
    1429: '9999963_00000_d_0000057.jpg', 1430: '9999947_00000_d_0000025.jpg', 1431: '9999938_00000_d_0000270.jpg',
    1432: '9999938_00000_d_0000471.jpg', 1433: '9999973_00000_d_0000078.jpg', 1434: '9999973_00000_d_0000048.jpg',
    1435: '9999938_00000_d_0000006.jpg', 1436: '0000320_00900_d_0000008.jpg', 1437: '0000006_05208_d_0000014.jpg',
    1438: '9999938_00000_d_0000055.jpg', 1439: '9999938_00000_d_0000400.jpg', 1440: '9999952_00000_d_0000219.jpg',
    1441: '9999952_00000_d_0000101.jpg', 1442: '9999952_00000_d_0000208.jpg', 1443: '9999938_00000_d_0000047.jpg',
    1444: '9999952_00000_d_0000034.jpg', 1445: '9999952_00000_d_0000118.jpg', 1446: '9999973_00000_d_0000121.jpg',
    1447: '9999979_00000_d_0000032.jpg', 1448: '9999973_00000_d_0000060.jpg', 1449: '9999938_00000_d_0000506.jpg',
    1450: '9999938_00000_d_0000015.jpg', 1451: '9999986_00000_d_0000025.jpg', 1452: '9999938_00000_d_0000145.jpg',
    1453: '9999973_00000_d_0000001.jpg', 1454: '0000282_00000_d_0000001.jpg', 1455: '9999996_00000_d_0000024.jpg',
    1456: '9999952_00000_d_0000278.jpg', 1457: '9999938_00000_d_0000225.jpg', 1458: '9999941_00000_d_0000003.jpg',
    1459: '0000272_05000_d_0000011.jpg', 1460: '9999938_00000_d_0000208.jpg', 1461: '9999952_00000_d_0000105.jpg',
    1462: '9999952_00000_d_0000066.jpg', 1463: '9999938_00000_d_0000430.jpg', 1464: '9999952_00000_d_0000121.jpg',
    1465: '9999963_00000_d_0000011.jpg', 1466: '9999973_00000_d_0000025.jpg', 1467: '0000087_00299_d_0000002.jpg',
    1468: '9999996_00000_d_0000007.jpg', 1469: '9999938_00000_d_0000121.jpg', 1470: '9999952_00000_d_0000185.jpg',
    1471: '9999986_00000_d_0000054.jpg', 1472: '9999938_00000_d_0000130.jpg', 1473: '9999976_00000_d_0000028.jpg',
    1474: '9999952_00000_d_0000067.jpg', 1475: '9999938_00000_d_0000128.jpg', 1476: '9999976_00000_d_0000026.jpg',
    1477: '9999938_00000_d_0000045.jpg', 1478: '9999938_00000_d_0000404.jpg', 1479: '9999938_00000_d_0000356.jpg',
    1480: '9999996_00000_d_0000013.jpg', 1481: '9999938_00000_d_0000346.jpg', 1482: '0000127_00002_d_0000119.jpg',
    1483: '9999938_00000_d_0000256.jpg', 1484: '9999952_00000_d_0000184.jpg', 1485: '0000310_02500_d_0000123.jpg',
    1486: '9999952_00000_d_0000186.jpg', 1487: '9999986_00000_d_0000009.jpg', 1488: '0000192_00522_d_0000213.jpg',
    1489: '9999952_00000_d_0000124.jpg', 1490: '9999979_00000_d_0000038.jpg', 1491: '9999973_00000_d_0000140.jpg',
    1492: '9999938_00000_d_0000420.jpg', 1493: '9999979_00000_d_0000044.jpg', 1494: '9999976_00000_d_0000006.jpg',
    1495: '9999963_00000_d_0000050.jpg', 1496: '9999938_00000_d_0000131.jpg', 1497: '9999938_00000_d_0000408.jpg',
    1498: '9999986_00000_d_0000056.jpg', 1499: '9999963_00000_d_0000019.jpg', 1500: '9999938_00000_d_0000413.jpg',
    1501: '9999938_00000_d_0000080.jpg', 1502: '9999938_00000_d_0000381.jpg', 1503: '9999973_00000_d_0000033.jpg',
    1504: '0000063_06000_d_0000007.jpg', 1505: '9999952_00000_d_0000022.jpg', 1506: '9999979_00000_d_0000020.jpg',
    1507: '9999938_00000_d_0000254.jpg', 1508: '0000120_02001_d_0000096.jpg', 1509: '9999996_00000_d_0000005.jpg',
    1510: '0000011_08299_d_0000012.jpg', 1511: '0000063_11000_d_0000012.jpg', 1512: '0000186_01387_d_0000188.jpg',
    1513: '0000006_01659_d_0000004.jpg', 1514: '9999952_00000_d_0000021.jpg', 1515: '9999952_00000_d_0000071.jpg',
    1516: '0000320_00300_d_0000004.jpg', 1517: '9999938_00000_d_0000367.jpg', 1518: '9999938_00000_d_0000298.jpg',
    1519: '9999979_00000_d_0000021.jpg', 1520: '9999938_00000_d_0000281.jpg', 1521: '9999973_00000_d_0000126.jpg',
    1522: '0000078_03810_d_0000011.jpg', 1523: '9999952_00000_d_0000282.jpg', 1524: '9999938_00000_d_0000474.jpg',
    1525: '9999938_00000_d_0000174.jpg', 1526: '0000063_10000_d_0000011.jpg', 1527: '9999986_00000_d_0000020.jpg',
    1528: '0000078_05467_d_0000015.jpg', 1529: '9999979_00000_d_0000002.jpg', 1530: '9999938_00000_d_0000374.jpg',
    1531: '9999973_00000_d_0000084.jpg', 1532: '9999938_00000_d_0000027.jpg', 1533: '9999952_00000_d_0000036.jpg',
    1534: '9999986_00000_d_0000016.jpg', 1535: '9999952_00000_d_0000241.jpg', 1536: '9999952_00000_d_0000224.jpg',
    1537: '9999952_00000_d_0000069.jpg', 1538: '0000078_07819_d_0000023.jpg', 1539: '9999952_00000_d_0000284.jpg',
    1540: '9999952_00000_d_0000140.jpg', 1541: '9999938_00000_d_0000362.jpg', 1542: '9999941_00000_d_0000006.jpg',
    1543: '0000328_00600_d_0000022.jpg', 1544: '9999938_00000_d_0000350.jpg', 1545: '0000074_06746_d_0000013.jpg',
    1546: '9999963_00000_d_0000061.jpg', 1547: '9999952_00000_d_0000263.jpg', 1548: '9999973_00000_d_0000108.jpg',
    1549: '9999973_00000_d_0000057.jpg', 1550: '9999947_00000_d_0000024.jpg', 1551: '9999938_00000_d_0000119.jpg',
    1552: '9999952_00000_d_0000314.jpg', 1553: '9999952_00000_d_0000050.jpg', 1554: '9999938_00000_d_0000186.jpg',
    1555: '9999938_00000_d_0000438.jpg', 1556: '9999952_00000_d_0000232.jpg', 1557: '0000365_01000_d_0000225.jpg',
    1558: '9999938_00000_d_0000232.jpg', 1559: '9999938_00000_d_0000090.jpg', 1560: '9999986_00000_d_0000014.jpg',
    1561: '9999979_00000_d_0000007.jpg', 1562: '0000265_03000_d_0000007.jpg', 1563: '9999963_00000_d_0000047.jpg',
    1564: '0000087_00700_d_0000003.jpg', 1565: '9999952_00000_d_0000154.jpg', 1566: '9999938_00000_d_0000122.jpg',
    1567: '9999938_00000_d_0000148.jpg', 1568: '9999952_00000_d_0000134.jpg', 1569: '9999963_00000_d_0000018.jpg',
    1570: '9999973_00000_d_0000089.jpg', 1571: '9999952_00000_d_0000197.jpg', 1572: '9999938_00000_d_0000038.jpg',
    1573: '9999938_00000_d_0000214.jpg', 1574: '9999986_00000_d_0000036.jpg', 1575: '9999938_00000_d_0000477.jpg',
    1576: '0000074_16210_d_0000032.jpg', 1577: '9999979_00000_d_0000009.jpg', 1578: '9999973_00000_d_0000064.jpg',
    1579: '9999993_00000_d_0000019.jpg', 1580: '9999963_00000_d_0000041.jpg', 1581: '0000120_03922_d_0000100.jpg',
    1582: '9999938_00000_d_0000418.jpg', 1583: '9999952_00000_d_0000041.jpg', 1584: '9999941_00000_d_0000012.jpg',
    1585: '9999938_00000_d_0000116.jpg', 1586: '9999952_00000_d_0000160.jpg', 1587: '9999973_00000_d_0000004.jpg',
    1588: '9999938_00000_d_0000084.jpg', 1589: '9999938_00000_d_0000512.jpg', 1590: '9999952_00000_d_0000213.jpg',
    1591: '9999938_00000_d_0000192.jpg', 1592: '9999938_00000_d_0000452.jpg', 1593: '0000259_00500_d_0000002.jpg',
    1594: '9999938_00000_d_0000022.jpg', 1595: '9999973_00000_d_0000034.jpg', 1596: '9999952_00000_d_0000035.jpg',
    1597: '9999938_00000_d_0000360.jpg', 1598: '9999952_00000_d_0000239.jpg', 1599: '9999986_00000_d_0000019.jpg',
    1600: '9999947_00000_d_0000008.jpg', 1601: '9999973_00000_d_0000062.jpg', 1602: '0000265_05000_d_0000011.jpg',
    1603: '9999938_00000_d_0000217.jpg', 1604: '9999952_00000_d_0000246.jpg', 1605: '9999952_00000_d_0000040.jpg',
    1606: '9999938_00000_d_0000159.jpg', 1607: '9999938_00000_d_0000063.jpg', 1608: '9999952_00000_d_0000011.jpg',
    1609: '0000370_00000_d_0000250.jpg', 1610: '9999938_00000_d_0000155.jpg',
}

## 2. Helpers

In [ ]:
# ===== 2. Helpers (this notebook uses the paper's own data, not the 498R All-in-One set) =========================
import common
DET = DET_H = REC = REC_H = None
common.save_json(common.env_info(), f"{WORK}/environment.json")

# ---- job helpers: each returns one command line; RUN starts them (one queue per GPU) and shows failures
PY = sys.executable
OCRMAX = ["--ocr-max", "40"] if FAST else []
SEQMAX = ["--max-test", "40"] if FAST else []
def DETJOB(name, model, epochs, data=None, imgsz=640, batch=16, hours=None, extra=()):
    return [PY, f"{KIT}/det.py", "--data", data or DET, "--model", model, "--name", name, "--project", RUNS,
            "--epochs", E(epochs), "--imgsz", imgsz, "--batch", batch, "--fraction", FR, *TMAX,
            *(["--max-hours", hours] if hours and not FAST else []), *extra]
def RECJOB(name, model, epochs, data=None, imgsz=320, batch=32, hours=None, extra=()):
    return [PY, f"{KIT}/rec.py", "--data", data or REC, "--model", model, "--name", name, "--project", RUNS,
            "--epochs", E(epochs), "--imgsz", imgsz, "--batch", batch, "--fraction", FR, *OCRMAX,
            *(["--max-hours", hours] if hours and not FAST else []), *extra]
def SEQJOB(name, model, epochs, extra=()):
    return [PY, f"{KIT}/seq.py", "--model", model, "--name", name, "--project", RUNS, "--epochs", E(epochs),
            *SEQMAX, *(["--real-fraction", "0.1"] if FAST else []), *extra]
def VEDJOB(name, epochs, extra=()):
    return [PY, f"{KIT}/ved.py", "--name", name, "--project", RUNS, "--epochs", E(epochs),
            *(["--max-train", "200", "--max-test", "20"] if FAST else []), *extra]
def SPEED(name, data=None, imgsz=640, trt=None, rtdetr=False):
    """batch-1 latency of a trained detector: PyTorch FP32/FP16 (+ TensorRT FP16 / INT8) on the T4, ONNX on the CPU."""
    t = [] if (FAST or not trt) else (["--trt"] if trt == "fp16" else ["--trt", trt])
    return [PY, f"{KIT}/det.py", "--data", data or DET, "--name", name, "--project", RUNS, "--imgsz", imgsz,
            "--fps-only", *t, *(["--rtdetr"] if rtdetr else [])]
def X(script, *args):
    return [PY, f"{KIT}/{script}", *args]
def RUN(jobs):
    """jobs: (name, gpu, cmd[, env]). Same gpu -> one after the other; different gpus -> at the same time."""
    jobs = [(j[0], j[1], [str(c) for c in j[2]]) + tuple(j[3:]) for j in jobs]
    codes = common.run_on_gpus(jobs, f"{WORK}/logs")
    for n, c in codes.items():
        if c != 0:
            print(f"----- {n} failed (code {c}); last lines of logs/{n}.log:")
            print(common.tail(f"{WORK}/logs/{n}.log", 25))
    return codes

## 3. VisDrone test-dev labels and the authors' result files

In [ ]:
VD = f"{WORK}/visdrone"
r = subprocess.run([PY, f"{KIT}/c36_sahi.py", "prep", "--root", VD, *(["--max", "3"] if FAST else [])],
                   capture_output=True, encoding="utf-8", errors="replace")
print(r.stdout[-2000:], r.stderr[-1500:])
assert r.returncode == 0, "STOP: the VisDrone labels or the authors' result files could not be downloaded (Internet on?)"

## 4. Re-score every file (CPU)

In [ ]:
try:
    RUN([("visdrone_rescore", None, X("c36_sahi.py", "eval", "--root", VD, "--name", "visdrone_rescore", "--project", RUNS))])
    d = common.load_json(f"{RUNS}/visdrone_rescore/results.json")
    for s, v in d["settings"].items(): print(f"{s:22s} paper {v['paper']['AP50']:5} | eval.json {v['authors_eval_json']['AP50']:5} | re-scored {v['rescored']}")
    print("abstract vs table:", json.dumps(d["abstract_vs_table"], indent=1))
except Exception:          # keep going: the results table and the outputs zip must still be written
    import traceback; traceback.print_exc()

## The paper's numbers vs. ours on the same data, and the outputs zip

In [ ]:
import report, shutil
rows = [('FCOS + FI: AP50 (%)', '25.8', 'p.3, Table 1', 'visdrone_rescore::settings.fcos_fi.rescored.AP50'), ('FCOS + SAHI + PO: AP50 (%)', '29.0', 'p.3, Table 1', 'visdrone_rescore::settings.fcos_sahi_po.rescored.AP50'), ('FCOS + SAHI + FI + PO: AP50 (%)', '31.0', 'p.3, Table 1', 'visdrone_rescore::settings.fcos_sahi_fi_po.rescored.AP50'), ('FCOS + SF + SAHI + PO: AP50 (%)', '38.1', 'p.3, Table 1', 'visdrone_rescore::settings.fcos_sf_sahi_po.rescored.AP50'), ('FCOS + SF + SAHI + FI + PO: AP50 (%)', '38.5', 'p.3, Table 1', 'visdrone_rescore::settings.fcos_sf_sahi_fi_po.rescored.AP50'), ('VFNet + FI: AP50 (%)', '28.8', 'p.3, Table 1', 'visdrone_rescore::settings.vfnet_fi.rescored.AP50'), ('VFNet + SAHI + PO: AP50 (%)', '32.0', 'p.3, Table 1', 'visdrone_rescore::settings.vfnet_sahi_po.rescored.AP50'), ('VFNet + SAHI + FI + PO: AP50 (%)', '33.9', 'p.3, Table 1', 'visdrone_rescore::settings.vfnet_sahi_fi_po.rescored.AP50'), ('VFNet + SF + SAHI + PO: AP50 (%)', '41.9', 'p.3, Table 1', 'visdrone_rescore::settings.vfnet_sf_sahi_po.rescored.AP50'), ('VFNet + SF + SAHI + FI + PO: AP50 (%)', '42.2', 'p.3, Table 1', 'visdrone_rescore::settings.vfnet_sf_sahi_fi_po.rescored.AP50'), ('TOOD + FI: AP50 (%)', '29.4', 'p.3, Table 1', 'visdrone_rescore::settings.tood_fi.rescored.AP50'), ('TOOD + SAHI: AP50 (%)', '31.9', 'p.3, Table 1', 'visdrone_rescore::settings.tood_sahi.rescored.AP50'), ('TOOD + SAHI + PO: AP50 (%)', '32.5', 'p.3, Table 1', 'visdrone_rescore::settings.tood_sahi_po.rescored.AP50'), ('TOOD + SAHI + FI: AP50 (%)', '34.6', 'p.3, Table 1', 'visdrone_rescore::settings.tood_sahi_fi.rescored.AP50'), ('TOOD + SAHI + FI + PO: AP50 (%)', '34.7', 'p.3, Table 1', 'visdrone_rescore::settings.tood_sahi_fi_po.rescored.AP50'), ('TOOD + SF + FI: AP50 (%)', '36.8', 'p.3, Table 1', 'visdrone_rescore::settings.tood_sf_fi.rescored.AP50'), ('TOOD + SF + SAHI: AP50 (%)', '42.5', 'p.3, Table 1', 'visdrone_rescore::settings.tood_sf_sahi.rescored.AP50'), ('TOOD + SF + SAHI + PO: AP50 (%)', '43.1', 'p.3, Table 1', 'visdrone_rescore::settings.tood_sf_sahi_po.rescored.AP50'), ('TOOD + SF + SAHI + FI: AP50 (%)', '43.4', 'p.3, Table 1', 'visdrone_rescore::settings.tood_sf_sahi_fi.rescored.AP50'), ('TOOD + SF + SAHI + FI + PO: AP50 (%)', '43.5', 'p.3, Table 1', 'visdrone_rescore::settings.tood_sf_sahi_fi_po.rescored.AP50'), ('TOOD + FI: AP50 small (%)', '18.1', 'p.3, Table 1', 'visdrone_rescore::settings.tood_fi.rescored.AP50_s'), ('TOOD + SF + SAHI + FI + PO: AP50 small (%)', '31.7', 'p.3, Table 1', 'visdrone_rescore::settings.tood_sf_sahi_fi_po.rescored.AP50_s'), ('FCOS: SAHI gain over FI, AP50 (pp)', '6.8 (abstract)', 'p.1', 'visdrone_rescore::abstract_vs_table.fcos.table_sahi_gain'), ('FCOS: SF + SAHI gain over FI (pp)', '12.7 (abstract)', 'p.1', 'visdrone_rescore::abstract_vs_table.fcos.table_sf_gain'), ('VFNet: SAHI / SF + SAHI gain (pp)', '5.1 / 13.4 (abstract)', 'p.1', 'visdrone_rescore::abstract_vs_table.vfnet.table_sf_gain'), ('TOOD: SAHI gain over FI (pp)', '5.3 (abstract)', 'p.1', 'visdrone_rescore::abstract_vs_table.tood.table_sahi_gain'), ('TOOD: SF + SAHI gain over FI (pp)', '14.5 (abstract)', 'p.1', 'visdrone_rescore::abstract_vs_table.tood.table_sf_gain'), ('Largest |re-scored - Table 1| AP50 (pp)', '-', '', 'visdrone_rescore::max_abs_diff_rescored_vs_paper_AP50')]
md = report.compare_table(RUNS, rows, ['visdrone_rescore'], "Paper #36 on its own data: ours vs. the paper",
                          "Same data and the authors' own predictions: a direct check of Table 1. 'SAHI gain' = best SAHI row without SF minus FI; 'SF + SAHI gain' = best row of that detector minus FI (the abstract's wording).", same_data=True)
open(f"{WORK}/results_original_check.md", "w", encoding="utf-8").write(md); print(md)
os.makedirs(f"{WORK}/code", exist_ok=True)
for f in glob.glob(f"{KIT}/*.py"): shutil.copy(f, f"{WORK}/code/")
report.pack(WORK, "paper36_check")

Download `paper36_check_outputs.zip` from the **Output** tab and upload it to the team Drive (`498R / weights / paper36 / original_check /`).